# CTDDG fine-tuning (stage 2)

Loads the stage-1 checkpoint, adds the protein conditioning path, and trains on BindingDB
protein-ligand pairs with the Eq. 8 toxicity-weighted loss. Self-contained; run top to bottom.

**Needs** `<fold>_<tr|te>_unique_embeddings_bepler.h5` from `01_generate_prose_embeddings.ipynb`:
one average-pooled `(6165,)` vector per protein, keyed `p1 … pN`. Per-residue output is rejected
on load. Pairs are assembled from the raw fold files, so no pre-built SMILES+embedding file is
needed.

**Produces** `ckpt_best.pt` (lowest validation loss, the one to evaluate), `ckpt_last.pt` (resume
point), `test_set.npz` and `split_manifest.json`. Re-running resumes from `ckpt_last.pt`;
`FORCE_RESTART = True` archives the old run and starts clean. With `LIMIT_PAIRS` set, the data
artifacts get `.smoke` names and are not the paper's dataset.

In [1]:
import os
import gc
import sys
import glob
import json
import signal
import math
import time
import pickle
import random
import hashlib
import warnings
from abc import ABCMeta, abstractmethod

import numpy as np
import pandas as pd
import networkx as nx
from scipy import sparse

import rdkit
from rdkit import Chem
from rdkit import RDLogger

RDLogger.DisableLog('rdApp.*')
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Sampler
from torch.autograd.function import once_differentiable

print(f"torch {torch.__version__} | rdkit {rdkit.__version__}")

torch 2.4.1+cu121 | rdkit 2026.03.6


## Config

Everything that varies per run. Nothing below this section should need editing.

In [2]:
FORCE_RESTART = False
FOLD = os.environ.get("CTDDG_FOLD", "d1")

GPU_ID = 0
CTX = torch.device(f"cuda:{GPU_ID}") if torch.cuda.is_available() else torch.device("cpu")
print(f"Device: {CTX}")

REPO_ROOT = os.environ.get("CTDDG_REPO_ROOT", "/home/muhamed/repo/CTDDG")
ATOM_TYPES_PATH = os.path.join(REPO_ROOT, "datasets/data/atom_types.txt")
BINDINGDB_DIR = os.path.join(REPO_ROOT, "datasets/data/bindingdb")
RF_MODEL_PATH = os.path.join(REPO_ROOT, "models/ames_random_forest.pkl")
FP_CONFIG_PATH = os.path.join(REPO_ROOT, "models/fingerprint_config.pkl")
PRETRAINED_CKPT = os.path.join(REPO_ROOT, "checkpoints/ckpt.pt")
# Stage 2 without stage 1 is not CTDDG. False only for a deliberate from-scratch run.
REQUIRE_PRETRAINED = True
# Stop if the checkpoint carries positive evidence of never having been trained (every
# BatchNorm reports num_batches_tracked == 0). True only to fine-tune one deliberately.
ALLOW_UNTRAINED_CKPT = False

OUT_DIR = os.path.join(REPO_ROOT, "outputs/finetune", FOLD)
os.makedirs(OUT_DIR, exist_ok=True)

CKPT_BEST = os.path.join(OUT_DIR, "ckpt_best.pt")      # evaluate this one
CKPT_LAST = os.path.join(OUT_DIR, "ckpt_last.pt")      # resume from this one
CKPT_TRAINER = os.path.join(OUT_DIR, "trainer.pt")
CKPT_CONFIG = os.path.join(OUT_DIR, "configs.json")
LOG_FILE = os.path.join(OUT_DIR, "log.out")
VAL_LOG_FILE = os.path.join(OUT_DIR, "val.out")
CKPT_PREV = CKPT_LAST + ".prev"      # rotated backup, used if ckpt_last.pt is unreadable
# SPLIT_MANIFEST and TEST_BUNDLE are named further down, once LIMIT_PAIRS is known: a
# limited run must not overwrite the full run's artifacts or be mistaken for one.

Device: cuda:0


In [3]:
# Checkpoint/resume behaviour
CHECKPOINT_EVERY = 200   # iterations between checkpoints (also saves at every epoch end)
VERIFY_FASTA_ORDER = True  # assert the bepler FASTA, the protein list and the h5 keys agree
GRACEFUL_SIGNALS = True    # SIGTERM/SIGINT -> checkpoint at the next iteration, then exit

# fold -> the `4_org_*` directory suffix on disk
FOLD_SUFFIX = {"d1": "4_org_1042_104", "d2": "4_org_1000_112", "d3": "4_org_1004_122",
               "d4": "4_org_1002_103", "d5": "4_org_1036_124"}

AMES_CACHE = os.path.join(BINDINGDB_DIR, "ligand_ames_labels.csv")
AMES_CACHE_META = AMES_CACHE + ".meta.json"

# If the Ames RF is missing or was pickled by an incompatible sklearn, rebuild it here by the
# same procedure as code/random_forest_classifier.ipynb and save it to RF_MODEL_PATH, so both
# training phases use one artifact rather than a throwaway copy.
RF_AUTO_TRAIN = True
RF_META_PATH = os.path.splitext(RF_MODEL_PATH)[0] + ".meta.json"
HANSEN_PATHS = ((os.path.join(REPO_ROOT, "datasets/Mutagenicity_N7090.csv"), "\t"),
                (os.path.join(REPO_ROOT, "datasets/Mutagenicity_N6512.csv"), ","))

In [4]:
# The paper evaluates on all 104 novel proteins of the held-out split (Tables 1-3), so the
# whole test_dataset is the test set and validation is carved out of the TRAIN split instead.
# It is carved by protein, not by pair: early stopping for a conditional model should measure
# generalisation to unseen proteins, which a validation set sharing proteins with training
# cannot do. Cost: the training split loses this fraction of its 1042 proteins. The paper
# specifies no validation protocol, so this is our choice, not its.
VAL_PROTEIN_FRACTION = 0.05
SPLIT_SEED = 42

MAX_ATOMS = 100          # preprocessing drops larger ligands
# Pool the pair files from both split directories and route rows by the fold's protein lists.
# Required here: the pair files are not partitioned the way the fold is (see read_pairs).
POOL_PAIR_FILES = True
# The ligands_*/proteins_*_corrected files are a SUPERSET of the fold: they hold pairs for
# proteins outside this split (including test proteins in the train directory). The fold's
# split is defined by <fold>_<tr|te>_unique_proteins.txt, so filtering pair rows down to the
# listed proteins is the split, not attrition - expect a large drop. The invariant that does
# matter is checked below: every listed protein must appear in the pair file.
# Set this below 1.0 to also cap the dropped fraction. diagnose_unmatched.py explains a run.
UNMATCHED_TOLERANCE = 1.0

# None = the full dataset, which is what the paper's numbers come from. A number is a SMOKE
# TEST: it samples that many pooled pairs so the pipeline can be exercised before GPU time is
# committed, and everything it produces is named and tagged as a smoke artifact.
LIMIT_PAIRS = None       # e.g. 20_000 for a smoke run
SMOKE_TEST = LIMIT_PAIRS is not None
_SMOKE_SUFFIX = ".smoke" if SMOKE_TEST else ""
SPLIT_MANIFEST = os.path.join(OUT_DIR, f"split_manifest{_SMOKE_SUFFIX}.json")
TEST_BUNDLE = os.path.join(OUT_DIR, f"test_set{_SMOKE_SUFFIX}.npz")

In [5]:
BATCH_SIZE = 8
BATCH_SIZE_VAL = 8
K_PATHS = 5
P_ALPHA = 0.8
LAMBDA_TOX = 1.0
N_C = 6165               # average-pooled ProSE/Bepler width; validated per protein on load
ALLOW_N_C_OVERRIDE = False   # True = adopt the embedding file's width instead of erroring
EMB_EXTENSIONS = ('.h5', '.hdf5', '.npy', '.txt', '.csv')

F_E = 16
F_H = [32, 64, 128, 128, 256, 256]
F_SKIP = 256
F_C = [512]
FH_POLICY = 128
ACTIVATION = 'relu'
N_RNN = 3
D_RECEPTIVE = 2
MAX_ITER = 120

LR = 1e-4
DECAY = 0.002            # lr *= (1 - DECAY) every DECAY_STEP iterations
DECAY_STEP = 100
CLIP_GRAD = 3.0
SUMMARY_STEP = 200
MAX_EPOCHS = 10
PATIENCE = 10
ES_DELTA = 1e-2

### Model selection and the toxic branch

Neither is in the paper. `EARLY_STOP_ON` picks what early stopping watches; `TOX_FLOOR = None`
is Eq. 8 exactly as published and the baseline for anything compared against the paper.

In [6]:
# --- Model selection -------------------------------------------------------------------
# Which validation quantity early stopping watches. This is NOT part of Eq. 8; the paper
# specifies no model-selection rule at all, so choosing one is ours either way.
#   'nll'      - plain -mean(l), bounded below by 0. DEFAULT.
#   'weighted' - the Eq. 8 objective itself, which is unbounded below on the toxic branch,
#                so "improvement" can mean the toxic branch diverging. Selecting the best
#                checkpoint by an unbounded quantity can pick the most degenerate one.
# Note LAMBDA_TOX = 0 does NOT recover CDGCN: the weight 1-(1+0)*tox is 0 on toxic ligands,
# so they are dropped rather than trained on normally. It is still the clean control for
# "is the toxic branch causing this?", since it removes the gradient reversal entirely.
EARLY_STOP_ON = 'nll'

# Warn when validation NLL rises this far above its own best.
NLL_DIVERGENCE_MARGIN = 2.0

# --- Toxic-branch floor (review item F13) ----------------------------------------------
# None = Eq. 8 exactly as published, and the paper-faithful baseline. Keep it for any run
# whose numbers you intend to compare against the paper.
#
# A float applies  l = max(l, TOX_FLOOR)  for toxic ligands only, in the OBJECTIVE. Reported
# metrics stay unfloored. Once a toxic molecule is below the floor, clamp passes zero
# gradient, so it stops being pushed further down - which is the point: the published toxic
# term is unbounded below.
#
# Flooring the per-molecule bound AFTER the importance-sampling reduction is the right place:
# it leaves the Eq. 3 estimate of log P(G|P) untouched and only bounds how far the objective
# will chase one molecule. Flooring the per-path terms before logsumexp would corrupt the
# estimator itself.
#
# Choosing the value is a judgement call, NOT something the data settles. A floor only binds
# while toxic likelihood is above it, so it must be set from the START of a run: applied to a
# run that has already driven toxic ligands far down, every toxic molecule is clamped from
# iteration 1 and the toxic branch is inert (equivalent to LAMBDA_TOX = 0). The log reports
# frac_floored so this is visible. As a starting point, a floor well below mean l_nontoxic
# (which was about -80 here) already makes toxic ligands astronomically unlikely: at -120 a
# toxic ligand is e^-40 as likely as a typical non-toxic one. Try the least aggressive floor
# that holds validation NLL steady.
TOX_FLOOR = None

In [7]:
NUM_WORKERS = 8
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

_missing = []
for _label, _p in (("atom_types.txt", ATOM_TYPES_PATH),
                   ("bindingdb dir", BINDINGDB_DIR),
                   ("fingerprint config", FP_CONFIG_PATH)):
    if not os.path.exists(_p):
        _missing.append(f"{_label}: {_p}")
if _missing:
    raise FileNotFoundError("Required inputs missing:\n  " + "\n  ".join(_missing))

# The Ames RF is only needed when a ligand is not already in the label cache, so it is
# checked lazily in ames_labels() rather than blocking startup. It is gitignored in this
# repo; re-running the training cells of code/random_forest_classifier.ipynb regenerates
# it deterministically (random_state=42).
if not os.path.exists(RF_MODEL_PATH):
    print(f"[WARN] No Ames RF at {RF_MODEL_PATH}. The label cache can only be reused when "
          f"the model that wrote it is on disk to check against, so it will be retrained "
          f"and every ligand relabelled. Regenerate it from "
          f"code/random_forest_classifier.ipynb to avoid that.")

# Fail here, not after data assembly and RF labelling have already spent CPU hours.
if not os.path.exists(PRETRAINED_CKPT):
    if REQUIRE_PRETRAINED:
        raise FileNotFoundError(
            f"No pretrained checkpoint at {PRETRAINED_CKPT}.\n"
            f"  CTDDG is a two-phase method (paper 2.4): stage 1 pretrains unconditionally on "
            f"ChEMBL, stage 2 fine-tunes that model. Starting from random weights is a "
            f"different experiment and its numbers are not comparable to the paper.\n"
            f"  Run stage-1 pretraining first, or set REQUIRE_PRETRAINED = False to train "
            f"from scratch deliberately.")
    print("[WARN] REQUIRE_PRETRAINED is off and no checkpoint is present - this run will "
          "train from random init, which is NOT the two-phase protocol.")

if SMOKE_TEST:
    print(f"\n[SMOKE TEST] LIMIT_PAIRS = {LIMIT_PAIRS}. Outputs are written as "
          f"{os.path.basename(SPLIT_MANIFEST)} / {os.path.basename(TEST_BUNDLE)} and are "
          f"NOT the paper's dataset. Set LIMIT_PAIRS = None for the real run.\n")
print("[OK] Paths verified.")

[OK] Paths verified.


## Restart cleanup

Runs before any data work. A crash during assembly or RF labelling must not leave a resumable
checkpoint from the previous run sitting there for the next one to pick up.

In [8]:
RESTART_STAMP = time.strftime('%Y%m%d-%H%M%S')


def archive_stale(paths, stamp=None, why=""):
    """Move previous-run artifacts aside as <name>.stale.<stamp>. -> list of new paths."""
    stamp = stamp or RESTART_STAMP
    moved = []
    for p in paths:
        if os.path.exists(p):
            dst = f"{p}.stale.{stamp}"
            os.replace(p, dst)
            moved.append(dst)
            print(f"[INFO] Moved {os.path.basename(p)} aside -> {os.path.basename(dst)}"
                  + (f"  ({why})" if why else ""))
    return moved


# Checkpoints and logs move together, so the two can never describe different runs.
RESUMABLE_ARTIFACTS = (CKPT_BEST, CKPT_LAST, CKPT_PREV, CKPT_TRAINER, LOG_FILE, VAL_LOG_FILE)

if FORCE_RESTART:
    # Here, not in the model cell: a run that dies in assemble_pairs or RF training would
    # otherwise leave ckpt_last.pt in place, and the next run with FORCE_RESTART back off
    # would silently resume the OLD run while reporting a fresh start.
    print("[INFO] FORCE_RESTART is set - archiving the previous run before any data work.")
    archive_stale(RESUMABLE_ARTIFACTS, why="FORCE_RESTART")

## Stage-1 provenance

What the files on disk actually say about `ckpt.pt`. A state dict that loads cleanly proves the
shapes match, not that any training happened.

In [9]:
def _last_log_step(path):
    """-> (last_step, n_data_rows) from a stage-1 log.out, or (None, 0)."""
    last, n = None, 0
    try:
        with open(path) as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith(('#', 'step\t')) or line == 'Training finished':
                    continue
                try:
                    last = int(line.split('\t')[0])
                    n += 1
                except (ValueError, IndexError):
                    continue
    except OSError:
        return None, 0
    return last, n

In [10]:
def pretrained_provenance(path):
    """Report what the files on disk actually say about this checkpoint.

    Evidence only - nothing here is inferred. A state dict that loads cleanly proves the
    SHAPES match, not that any training happened, so loading is deliberately not counted.
    The strongest available signal is BatchNorm's num_batches_tracked, which counts
    train-mode forward passes and is reset to 0 by init_weights_xavier.
    """
    ev = {'path': path, 'size_mb': None, 'mtime': None, 'bn_steps': None,
          'trainer_counter': None, 'log_last_step': None, 'log_rows': 0, 'trained': False}
    ckpt_dir = os.path.dirname(path)

    try:
        st = os.stat(path)
        ev['size_mb'] = st.st_size / 1e6
        ev['mtime'] = time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(st.st_mtime))
    except OSError:
        return ev

    try:
        state = torch.load(path, map_location='cpu', weights_only=False)
        sd = state['model'] if isinstance(state, dict) and 'model' in state else state
        nbt = [int(v) for k, v in sd.items() if k.endswith('num_batches_tracked')]
        ev['bn_steps'] = max(nbt) if nbt else None
    except Exception as e:
        print(f"[WARN] could not read {os.path.basename(path)}: {type(e).__name__}: {e}")

    tr = os.path.join(ckpt_dir, 'trainer.pt')
    if os.path.isfile(tr):
        try:
            t = torch.load(tr, map_location='cpu', weights_only=False)
            if isinstance(t, dict):
                ev['trainer_counter'] = t.get('global_counter')
        except Exception:
            pass

    ev['log_last_step'], ev['log_rows'] = _last_log_step(os.path.join(ckpt_dir, 'log.out'))

    ev['trained'] = bool((ev['bn_steps'] or 0) > 0
                         or (ev['trainer_counter'] or 0) > 0
                         or ev['log_rows'] > 0)

    print(f"\n--- stage-1 checkpoint provenance: {os.path.basename(path)} ---")
    print(f"  size / modified     {ev['size_mb']:.1f} MB  {ev['mtime']}")
    print(f"  BN train-mode steps {ev['bn_steps'] if ev['bn_steps'] is not None else 'absent'}"
          f"   (0 = never trained; counts forward passes, so a lower bound on iterations)")
    print(f"  trainer.pt counter  {ev['trainer_counter'] if ev['trainer_counter'] is not None else 'absent'}")
    print(f"  log.out             {ev['log_rows']} data rows"
          + (f", last step {ev['log_last_step']}" if ev['log_last_step'] is not None else ""))
    if ev['trained']:
        print("  -> trained: yes, by the evidence above.")
    else:
        print("  -> [UNVERIFIED] nothing on disk shows this checkpoint was ever trained.")
    print("  Not verifiable from this file either way: which objective stage 1 used. "
          "ckpt.pt stores weights only - no lambda, no Eq. 8 flag. A CDGCN-objective "
          "checkpoint is indistinguishable from a CTDDG one here.")
    print("-" * 60)
    return ev

In [11]:
PRETRAIN_EVIDENCE = None
if os.path.exists(PRETRAINED_CKPT):
    PRETRAIN_EVIDENCE = pretrained_provenance(PRETRAINED_CKPT)
    if PRETRAIN_EVIDENCE['bn_steps'] == 0 and not ALLOW_UNTRAINED_CKPT:
        raise RuntimeError(
            f"{PRETRAINED_CKPT} reports num_batches_tracked == 0 in every BatchNorm layer: "
            f"it has never seen a training step. Fine-tuning it is fine-tuning random "
            f"weights, at full GPU cost.\n"
            f"  Run stage-1 pretraining first, or set ALLOW_UNTRAINED_CKPT = True if this "
            f"is deliberate.")
    if not PRETRAIN_EVIDENCE['trained']:
        print("[UNVERIFIED] Proceeding without evidence that stage 1 ran. If this checkpoint "
              "is a smoke-test artifact, the fine-tuning results will not mean anything.")


--- stage-1 checkpoint provenance: ckpt.pt ---
  size / modified     28.0 MB  2026-09-30 13:26:50
  BN train-mode steps 480000   (0 = never trained; counts forward passes, so a lower bound on iterations)
  trainer.pt counter  480000
  log.out             5433 data rows, last step 480000
  -> trained: yes, by the evidence above.
  Not verifiable from this file either way: which objective stage 1 used. ckpt.pt stores weights only - no lambda, no Eq. 8 flag. A CDGCN-objective checkpoint is indistinguishable from a CTDDG one here.
------------------------------------------------------------


## Fold files and embeddings

Row `i` of the embedding matrix is HDF5 key `p{i+1}` is FASTA record `i`. Pairs are matched to
rows by protein sequence, so the FASTA and the protein list have to agree record for record.
Both are checked, not assumed.

In [12]:
def split_dir(split):
    """split: 'train' or 'test'."""
    return os.path.join(BINDINGDB_DIR, f"{split}_dataset", f"{split}_{FOLD_SUFFIX[FOLD]}")


def find_fold_files(split):
    """Locate the four files one split needs: ligands, proteins, unique proteins, embeddings."""
    d = split_dir(split)
    tag = "tr" if split == "train" else "te"

    def one(pattern, what):
        hits = sorted(glob.glob(os.path.join(d, pattern)))
        if not hits:
            raise FileNotFoundError(
                f"{what}: no match for {pattern!r} in {d}\n"
                f"  Directory holds: {sorted(os.listdir(d))[:25]}")
        return hits[0]

    ligands = one("ligands_*_corrected", "ligand pairs")
    proteins = one("proteins_*_corrected", "protein pairs")
    unique = one(f"{FOLD}_{tag}_unique_proteins.txt", "unique protein list")

    # exact name first, then any other embedding file in the directory
    expected = f"{FOLD}_{tag}_unique_embeddings_bepler.h5"
    emb = os.path.join(d, expected)
    if not os.path.isfile(emb):
        emb_hits = [p for p in sorted(glob.glob(os.path.join(d, "*embeddings*bepler*")))
                    if os.path.splitext(p)[1].lower() in EMB_EXTENSIONS]
        if not emb_hits:
            present = sorted(os.listdir(d)) if os.path.isdir(d) else []
            raise FileNotFoundError(
                f"Protein embeddings missing for fold {FOLD} ({split} split).\n"
                f"  Expected: {os.path.join(d, expected)}\n"
                f"  Directory holds: {present[:25]}\n"
                f"  Generate average-pooled embeddings with:\n"
                f"    python prose/embed_sequences.py --pool avg \\\n"
                f"        -o {expected} {FOLD}_{tag}_unique_proteins_bepler.fa\n"
                f"  --pool avg is required: it yields one ({N_C},) vector per protein. "
                f"--pool none yields per-residue (L, {N_C}) arrays, which this code rejects.")
        emb = emb_hits[0]
        print(f"[WARN] {expected} not found; using {os.path.basename(emb)} instead.")
    return ligands, proteins, unique, emb

In [13]:
def _as_pooled_vector(arr, name, path, width):
    """Validate one protein's embedding and return it as 1-D float32.

    `width` is the required length, or None to accept whatever the first protein set.
    """
    arr = np.asarray(arr)

    if arr.ndim != 1:
        squeezed = arr.squeeze()
        if squeezed.ndim == 1:
            arr = squeezed                       # (1, N_C) / (N_C, 1) are fine
        elif arr.ndim == 2:
            raise ValueError(
                f"{path}: protein {name!r} has shape {arr.shape} — per-residue embeddings.\n"
                f"  This file was written with `--pool none`. Fine-tuning conditions on ONE "
                f"fixed-size vector per protein, so regenerate it with:\n"
                f"    python prose/embed_sequences.py --pool avg -o <out>.h5 <in>.fa\n"
                f"  Expected per protein: ({N_C},)   got: {arr.shape}")
        else:
            raise ValueError(f"{path}: protein {name!r} has shape {arr.shape}; "
                             f"expected a 1-D ({N_C},) vector.")

    if width is not None and arr.shape[0] != width:
        hint = ""
        if arr.shape[0] % N_C == 0 and arr.shape[0] > N_C:
            hint = (f"\n  {arr.shape[0]} = {arr.shape[0] // N_C} x {N_C}, so this looks like a "
                    f"flattened per-residue (`--pool none`) embedding. Regenerate with "
                    f"`--pool avg`.")
        raise ValueError(
            f"{path}: protein {name!r} has width {arr.shape[0]}; expected {width}.{hint}")

    if arr.shape[0] == 0:
        raise ValueError(f"{path}: protein {name!r} has an empty embedding.")

    return arr.astype(np.float32, copy=False)


def _h5_key_order(f, n_expected, path):
    """Dataset keys in protein order: p1, p2, ... pN as written by embed_sequences.py."""
    keys = list(f.keys())
    if not keys:
        raise ValueError(f"{path}: HDF5 file contains no datasets.")

    if len(keys) != n_expected:
        shown = keys[:5] if len(keys) <= 10 else keys[:5] + ['...'] + keys[-5:]
        raise ValueError(
            f"{path}: {len(keys)} embedded proteins but {n_expected} unique proteins in the "
            f"fold's protein list.\n"
            f"  Keys: {shown}\n"
            f"  The two must correspond one-to-one, in order. Re-run embed_sequences.py "
            f"--pool avg on this fold's *_unique_proteins_bepler.fa.")

    named = [f"p{i + 1}" for i in range(n_expected)]
    if all(k in f for k in named):
        return named

    # No fallback ordering. Row i of the embedding matrix must be protein i of the list,
    # and the only thing that guarantees that is the key name. Any other order silently
    # conditions every ligand on the wrong protein while all shape checks still pass.
    missing = [k for k in named if k not in f]
    raise ValueError(
        f"{os.path.basename(path)}: expected keys p1..p{n_expected}; {len(missing)} absent "
        f"(e.g. {missing[:8]}). Present keys look like {keys[:5]}.\n"
        f"  Regenerate the embeddings from {FOLD}_<tr|te>_unique_proteins_bepler.fa with "
        f"01_generate_prose_embeddings.ipynb, which writes exactly these keys.")

In [14]:
def load_embeddings(path, n_expected):
    """Read average-pooled ProSE embeddings into [n_proteins, N_C] float32.

    Row i is protein `p{i+1}`, which is the bepler FASTA header order and therefore the line
    order of `<fold>_<tr|te>_unique_proteins.txt`.
    """
    ext = os.path.splitext(path)[1].lower()
    if ext not in EMB_EXTENSIONS:
        raise ValueError(f"{path}: unsupported embedding format {ext!r}; "
                         f"expected one of {sorted(EMB_EXTENSIONS)}.")

    if ext in ('.h5', '.hdf5'):
        import h5py
        with h5py.File(path, 'r') as f:
            order = _h5_key_order(f, n_expected, path)

            # the first protein fixes the width every other protein must match
            first = _as_pooled_vector(f[order[0]][()], order[0], path,
                                      None if ALLOW_N_C_OVERRIDE else N_C)
            width = first.shape[0]
            rows = [first] + [_as_pooled_vector(f[k][()], k, path, width)
                              for k in order[1:]]
        C = np.stack(rows, axis=0)

    elif ext == '.npy':
        arr = np.load(path)
        if arr.ndim != 2:
            raise ValueError(
                f"{path}: expected a 2-D [n_proteins, {N_C}] array, got shape {arr.shape}. "
                f"Per-residue (`--pool none`) embeddings are not usable here.")
        C = arr.astype(np.float32, copy=False)

    else:
        raw = []
        with open(path) as f:
            for ln, line in enumerate(f, 1):
                parts = line.replace(',', ' ').split()
                if not parts:
                    continue
                try:
                    float(parts[0])
                except ValueError:
                    parts = parts[1:]            # leading sequence-name column
                raw.append((f"line {ln}", np.array(parts, dtype=np.float32)))
        if not raw:
            raise ValueError(f"{path}: no embedding rows parsed.")
        first = _as_pooled_vector(raw[0][1], raw[0][0], path,
                                  None if ALLOW_N_C_OVERRIDE else N_C)
        width = first.shape[0]
        C = np.stack([first] + [_as_pooled_vector(a, nm, path, width)
                                for nm, a in raw[1:]], axis=0)

    if C.shape[0] != n_expected:
        raise ValueError(
            f"{path}: {C.shape[0]} embedding rows but {n_expected} unique proteins in the "
            f"fold's protein list. The two must correspond one-to-one, in order.")

    if C.shape[1] != N_C:
        if not ALLOW_N_C_OVERRIDE:
            raise ValueError(
                f"{path}: embedding width {C.shape[1]}, expected N_C = {N_C}.\n"
                f"  Average-pooled ProSE/Bepler embeddings are {N_C}-d — either the wrong file "
                f"is being read or it was not produced with `--pool avg`.\n"
                f"  Set ALLOW_N_C_OVERRIDE = True to train at width {C.shape[1]} instead.")
        print(f"[WARN] N_C {N_C} -> {C.shape[1]} (ALLOW_N_C_OVERRIDE is on).")

    if not np.isfinite(C).all():
        n_bad = int((~np.isfinite(C)).any(axis=1).sum())
        raise ValueError(f"{path}: {n_bad} protein embeddings contain NaN or inf.")

    return C

## Ames labels

Ligand toxicity from the repo's Random Forest, cached across folds. The cache is reused only
when the model that wrote it is on disk and hashes the same; otherwise everything is relabelled.
Mixing labels from two forests would mean Eq. 8 optimising against two definitions of toxic.

In [15]:
def _load_pickle(path):
    """Load a pickle or joblib artifact, naming the usual cause when it fails."""
    errors = []
    for how, fn in (("pickle", lambda: pickle.load(open(path, 'rb'))),
                    ("joblib", lambda: __import__('joblib').load(path))):
        try:
            return fn()
        except Exception as e:
            errors.append(f"{how}: {type(e).__name__}: {e}")

    hint = ""
    if any("incompatible dtype" in e or "node array" in e for e in errors):
        import sklearn
        hint = (f"\n  This is a scikit-learn version mismatch, not a corrupt file: the pickle "
                f"was written by an older sklearn whose tree node dtype differs from "
                f"{sklearn.__version__} in this environment.\n"
                f"  Regenerate the model IN THIS ENVIRONMENT by re-running the training cells "
                f"of code/random_forest_classifier.ipynb (random_state=42). Do not downgrade "
                f"sklearn to match the old pickle.")
    raise RuntimeError(f"Could not load {path}\n  " + "\n  ".join(errors) + hint)


def morgan_matrix(smiles_list, radius, n_bits):
    """[n, n_bits] uint8 Morgan fingerprints. Rows for unparseable SMILES stay zero."""
    try:
        from rdkit.Chem import rdFingerprintGenerator
        gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)
        fp_of = lambda m: gen.GetFingerprint(m)
    except (ImportError, AttributeError):
        from rdkit.Chem import AllChem
        fp_of = lambda m: AllChem.GetMorganFingerprintAsBitVect(m, radius, nBits=n_bits)

    out = np.zeros((len(smiles_list), n_bits), dtype=np.uint8)
    for i, smi in enumerate(smiles_list):
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            out[i] = np.frombuffer(fp_of(mol).ToBitString().encode(), dtype=np.uint8) - ord('0')
        if (i + 1) % 20000 == 0:
            print(f"    fingerprinted {i + 1}/{len(smiles_list)}")
    return out

In [16]:
_RF_HASH_MEMO = {}


def _sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()


def rf_fingerprint():
    """Identifies the model+features a cached label came from.

    SHA-256 of the RF artifact, not size+mtime: a retrained forest can land on the same byte
    count, and a copy, checkout or rsync changes mtime without changing the model, so the old
    scheme could both miss a replacement and force pointless relabelling. Hashed once per
    (path, size, mtime) per process.

    Either half being None means 'cannot be verified' - never treated as a match.
    """
    rf = None
    try:
        st = os.stat(RF_MODEL_PATH)
        key = (RF_MODEL_PATH, st.st_size, st.st_mtime_ns)
        if key not in _RF_HASH_MEMO:
            _RF_HASH_MEMO[key] = _sha256(RF_MODEL_PATH)
        rf = {'sha256': _RF_HASH_MEMO[key], 'size': st.st_size}
    except OSError:
        rf = None

    fp = None
    try:
        cfg = _load_pickle(FP_CONFIG_PATH)
        fp = {'radius': int(cfg.get('radius', 2)),
              'n_bits': int(cfg.get('n_bits', cfg.get('nBits', 2048)))}
    except Exception:
        fp = None

    return {'v': 2, 'rf': rf, 'fp': fp}


def fingerprint_is_verifiable(fp):
    """A fingerprint with a missing half proves nothing and must not match anything."""
    return bool(fp) and bool(fp.get('rf')) and bool(fp.get('fp'))


def cache_decision(want, got, meta_err=None):
    """Should the on-disk label cache be reused? -> (accept, reason).

    Reuse requires a fingerprint that can be computed now AND an equal one recorded beside
    the cache. Missing, unreadable or unverifiable metadata means relabel, never reuse:
    labels from two different forests would make Eq. 8 optimise against two definitions of
    toxic, and nothing downstream would show it.
    """
    if not fingerprint_is_verifiable(want):
        if not (want or {}).get('rf'):
            return False, ("the Random Forest is not on disk, so the cache cannot be "
                           "checked against the model that wrote it")
        return False, "the fingerprint config is unreadable, so the cache cannot be checked"
    if got is None:
        return False, (f"fingerprint unreadable ({meta_err})" if meta_err
                       else "no fingerprint recorded beside the cache")
    if not fingerprint_is_verifiable(got):
        return False, "the recorded fingerprint is incomplete"
    if got != want:
        return False, "the recorded fingerprint differs from the current model/config"
    return True, "fingerprint matches"

In [17]:
def train_ames_rf(radius, n_bits):
    """Rebuild the Ames RF exactly as code/random_forest_classifier.ipynb does.

    Same merge and dedup of the two Hansen sets, same stratified 80/20 at random_state=42,
    same RandomForestClassifier(n_estimators=500, random_state=42). Fingerprints come from
    this notebook's morgan_matrix so the features the RF is trained on are bit-identical to
    the ones it will be asked to predict on.
    """
    import joblib
    import sklearn
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                                 roc_auc_score, confusion_matrix)
    from sklearn.model_selection import train_test_split

    frames = []
    for path, sep in HANSEN_PATHS:
        if not os.path.exists(path):
            raise FileNotFoundError(f"Hansen set missing: {path}")
        frames.append(pd.read_csv(path, sep=sep))
    df = pd.concat(frames, ignore_index=True)
    df = df.drop_duplicates()
    df = df.drop_duplicates(subset=["Canonical_Smiles"], keep="first").reset_index(drop=True)
    df = df.dropna(subset=["Canonical_Smiles", "Activity"]).reset_index(drop=True)
    df = df[df["Canonical_Smiles"].apply(lambda s: Chem.MolFromSmiles(s) is not None)]
    df = df.reset_index(drop=True)

    smiles = df["Canonical_Smiles"].astype(str).tolist()
    y = df["Activity"].astype(int).values
    X = morgan_matrix(smiles, radius, n_bits)
    print(f"[RF] {len(smiles)} molecules | {int(y.sum())} Ames positive")

    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, random_state=42,
                                              stratify=y)
    rf = RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)
    rf.fit(X_tr, y_tr)

    pred, prob = rf.predict(X_te), rf.predict_proba(X_te)[:, 1]
    meta = {"sklearn": sklearn.__version__, "n_molecules": len(smiles),
            "n_train": int(X_tr.shape[0]), "n_test": int(X_te.shape[0]),
            "radius": radius, "n_bits": n_bits,
            "accuracy": float(accuracy_score(y_te, pred)),
            "precision": float(precision_score(y_te, pred)),
            "recall": float(recall_score(y_te, pred)),
            "f1": float(f1_score(y_te, pred)),
            "roc_auc": float(roc_auc_score(y_te, prob)),
            "confusion_matrix": confusion_matrix(y_te, pred).tolist(),
            "trained_at": time.strftime("%Y-%m-%d %H:%M:%S")}
    print(f"[RF] accuracy {meta['accuracy']:.3f} | ROC-AUC {meta['roc_auc']:.3f} | "
          f"confusion {meta['confusion_matrix']}")

    os.makedirs(os.path.dirname(RF_MODEL_PATH), exist_ok=True)
    joblib.dump(rf, RF_MODEL_PATH)
    with open(RF_META_PATH, "w") as f:
        json.dump(meta, f, indent=1)
    print(f"[RF] saved {RF_MODEL_PATH}")
    print("[RF] NOTE: ChEMBL labels already on disk were produced by the previous RF. "
          "Re-run the ChEMBL annotation cells of code/random_forest_classifier.ipynb if you "
          "want both training phases labelled by this exact model.")
    return rf


def load_or_train_rf(radius, n_bits):
    """Load the Ames RF, rebuilding it if it is absent or unreadable by this sklearn."""
    if os.path.exists(RF_MODEL_PATH):
        try:
            return _load_pickle(RF_MODEL_PATH)
        except RuntimeError as e:
            if not RF_AUTO_TRAIN:
                raise
            print(f"[WARN] {RF_MODEL_PATH} could not be loaded:\n  {e}\n"
                  f"[WARN] RF_AUTO_TRAIN is on - retraining it here.")
    elif not RF_AUTO_TRAIN:
        raise FileNotFoundError(
            f"No Ames RF at {RF_MODEL_PATH} and RF_AUTO_TRAIN is off. Re-run the training "
            f"cells of code/random_forest_classifier.ipynb, or set RF_AUTO_TRAIN = True.")
    else:
        print(f"[INFO] No Ames RF at {RF_MODEL_PATH} - training it here.")
    return train_ames_rf(radius, n_bits)

In [18]:
def ames_labels(smiles_list, cache_path=AMES_CACHE):
    """SMILES -> {0,1} Ames mutagenicity from the repo's Random Forest. Cached across folds."""
    cache = {}
    want = rf_fingerprint()

    if cache_path and os.path.exists(cache_path):
        got, meta_err = None, None
        if os.path.exists(AMES_CACHE_META):
            try:
                with open(AMES_CACHE_META) as f:
                    got = json.load(f)
            except Exception as e:
                meta_err = type(e).__name__

        accept, reason = cache_decision(want, got, meta_err)
        if accept:
            df = pd.read_csv(cache_path)
            cache = dict(zip(df['smiles'].astype(str), df['ames'].astype(int)))
            print(f"[INFO] Ames cache: {len(cache)} entries from "
                  f"{os.path.basename(cache_path)} ({reason})")
        else:
            stamp = time.strftime('%Y%m%d-%H%M%S')
            archive_stale([cache_path, AMES_CACHE_META], stamp=stamp, why="unverified cache")
            print(f"[WARN] Not reusing the Ames cache: {reason}. Relabelling from scratch - "
                  f"mixing labels from two models would mean Eq. 8 optimises against two "
                  f"different definitions of toxic.")

    todo = sorted({s for s in smiles_list if s not in cache})
    if todo:
        fp_cfg = _load_pickle(FP_CONFIG_PATH)
        radius = int(fp_cfg.get('radius', 2))
        n_bits = int(fp_cfg.get('n_bits', fp_cfg.get('nBits', 2048)))
        print(f"[INFO] Labelling {len(todo)} new ligands | {fp_cfg.get('type', 'Morgan')} "
              f"radius {radius}, {n_bits} bits")

        rf = load_or_train_rf(radius, n_bits)

        # load_or_train_rf may have REPLACED the model on disk (unloadable pickle +
        # RF_AUTO_TRAIN). Anything already read from the cache came from the old forest, so
        # drop it rather than mix two models within one label set.
        now = rf_fingerprint()
        if cache and now != want:
            print(f"[WARN] the Random Forest was replaced while labelling; discarding the "
                  f"{len(cache)} cached labels and relabelling everything with the new model.")
            cache = {}
            todo = sorted(set(smiles_list))

        X = morgan_matrix(todo, radius, n_bits)
        if getattr(rf, 'n_features_in_', n_bits) != n_bits:
            raise ValueError(f"RF expects {rf.n_features_in_} features, fingerprint config "
                             f"gives {n_bits}.")
        pred = rf.predict(X).astype(int)
        cache.update(zip(todo, pred))

        if cache_path:
            pd.DataFrame({'smiles': list(cache.keys()), 'ames': list(cache.values())}) \
              .to_csv(cache_path, index=False)
            # Record the model that actually produced these labels, not the one seen on entry.
            with open(AMES_CACHE_META, 'w') as f:
                json.dump(rf_fingerprint(), f, indent=1)
            print(f"[INFO] Cache written: {os.path.basename(cache_path)}")

    labels = [int(cache[s]) for s in smiles_list]
    n_tox = sum(labels)
    print(f"[INFO] Ames: {n_tox} toxic / {len(labels) - n_tox} non-toxic "
          f"({100.0 * n_tox / max(1, len(labels)):.1f}% toxic)")
    return labels

## Molecule spec and traversal

Unchanged from the pretraining code. Both phases have to build graphs identically or the
transferred weights mean nothing.

In [19]:
class MoleculeSpec(object):
    def __init__(self, file_name=ATOM_TYPES_PATH):
        self.atom_types = []
        self.atom_symbols = []
        with open(file_name, 'r') as f:
            for line in f:
                atom_type_i = line.strip('\n').split(',')
                self.atom_types.append((atom_type_i[0], int(atom_type_i[1]), int(atom_type_i[2])))
                if atom_type_i[0] not in self.atom_symbols:
                    self.atom_symbols.append(atom_type_i[0])

        self.bond_orders = [Chem.BondType.AROMATIC, Chem.BondType.SINGLE,
                            Chem.BondType.DOUBLE, Chem.BondType.TRIPLE]
        self.max_iter = MAX_ITER

    def get_atom_type(self, atom):
        return self.atom_types.index(
            (atom.GetSymbol(), atom.GetFormalCharge(), atom.GetNumExplicitHs()))

    def get_bond_type(self, bond):
        return self.bond_orders.index(bond.GetBondType())

    def index_to_atom(self, idx):
        symbol, charge, hs = self.atom_types[idx]
        a = Chem.Atom(symbol)
        a.SetFormalCharge(charge)
        a.SetNumExplicitHs(hs)
        return a

    def index_to_bond(self, mol, begin_id, end_id, idx):
        mol.AddBond(begin_id, end_id, self.bond_orders[idx])

    @property
    def num_atom_types(self):
        return len(self.atom_types)

    @property
    def num_bond_types(self):
        return len(self.bond_orders)


_mol_spec = None


def get_mol_spec():
    global _mol_spec
    if _mol_spec is None:
        _mol_spec = MoleculeSpec(ATOM_TYPES_PATH)
    return _mol_spec


def get_graph_from_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES string: {smiles}")

    atom_types, atom_ranks, bonds, bond_types = [], [], [], []
    for a, r in zip(mol.GetAtoms(), Chem.CanonicalRankAtoms(mol)):
        atom_types.append(get_mol_spec().get_atom_type(a))
        atom_ranks.append(r)

    for b in mol.GetBonds():
        idx_1, idx_2 = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        bt = get_mol_spec().get_bond_type(b)
        bonds.append([idx_1, idx_2])
        bond_types.append(bt)

    graph = nx.Graph()
    graph.add_nodes_from(range(len(atom_types)))
    graph.add_edges_from(bonds)
    return graph, atom_types, atom_ranks, bonds, bond_types

In [20]:
def traverse_graph(graph, atom_ranks, current_node=None, step_ids=None, p=P_ALPHA, log_p=0.0):
    """Q_alpha proposal: canonical rank with probability p, random neighbour otherwise."""
    if current_node is None:
        next_nodes = range(len(atom_ranks))
        step_ids = [-1] * len(next_nodes)
        next_node_ranks = atom_ranks
    else:
        next_nodes = list(graph.neighbors(current_node))
        next_nodes = [n for n in next_nodes if step_ids[n] < 0]
        next_node_ranks = [atom_ranks[n] for n in next_nodes]

    next_nodes = [n for n, r in sorted(zip(next_nodes, next_node_ranks), key=lambda x: x[1])]

    while len(next_nodes) > 0:
        if len(next_nodes) == 1:
            next_node = next_nodes[0]
        elif random.random() >= (1.0 - p):
            next_node = next_nodes[0]
            log_p += np.log(p)
        else:
            next_node = next_nodes[random.randint(1, len(next_nodes) - 1)]
            log_p += np.log((1.0 - p) / (len(next_nodes) - 1))

        step_ids[next_node] = max(step_ids) + 1
        _, log_p = traverse_graph(graph, atom_ranks, next_node, step_ids, p, log_p)
        next_nodes = [n for n in graph.neighbors(next_node) if step_ids[n] < 0]
        next_node_ranks = [atom_ranks[n] for n in next_nodes]
        next_nodes = [n for n, r in sorted(zip(next_nodes, next_node_ranks), key=lambda x: x[1])]

    return step_ids, log_p

In [21]:
def single_reorder(X_0, A_0, step_ids):
    X_0, A_0 = np.copy(X_0), np.copy(A_0)
    step_ids = np.array(step_ids, dtype=np.int32)
    sorted_ids = np.argsort(step_ids)
    X_0 = X_0[sorted_ids]
    A_0[:, 0], A_0[:, 1] = step_ids[A_0[:, 0]], step_ids[A_0[:, 1]]

    max_b, min_b = np.amax(A_0[:, :2], axis=1), np.amin(A_0[:, :2], axis=1)
    A_0 = A_0[np.lexsort([-min_b, max_b]), :]

    max_b, min_b = np.amax(A_0[:, :2], axis=1), np.amin(A_0[:, :2], axis=1)
    is_append = np.concatenate([np.array([True]), max_b[1:] > max_b[:-1]])
    A_0 = np.concatenate([
        np.where(is_append[:, np.newaxis], np.stack([min_b, max_b], axis=1),
                 np.stack([max_b, min_b], axis=1)),
        A_0[:, -1:]
    ], axis=1)
    return X_0, A_0


def single_expand(X_0, A_0):
    X_0, A_0 = np.copy(X_0), np.copy(A_0)
    is_append_iter = np.less(A_0[:, 0], A_0[:, 1]).astype(np.int32)

    NX = np.cumsum(np.pad(is_append_iter, [[1, 0]], mode='constant', constant_values=1))
    shift = np.cumsum(np.pad(NX, [[1, 0]], mode='constant')[:-1])
    X_index = np.arange(NX.sum(), dtype=np.int32) - np.repeat(shift, NX)
    X = X_0[X_index]

    _, A_index = np.tril_indices(A_0.shape[0])
    A = A_0[A_index, :]
    NA = np.arange(A_0.shape[0] + 1)

    action_type = 1 - is_append_iter
    atom_type = np.where(action_type == 0, X_0[A_0[:, 1]], 0)
    bond_type = A_0[:, 2]
    append_pos = np.where(action_type == 0, A_0[:, 0], 0)
    connect_pos = np.where(action_type == 1, A_0[:, 1], 0)
    actions = np.stack([action_type, atom_type, bond_type, append_pos, connect_pos], axis=1)
    actions = np.append(actions, [[2, 0, 0, 0, 0]], axis=0)
    action_0 = np.array([X_0[0]], dtype=np.int32)

    last_atom_index = shift + NX - 1
    last_atom_mask = np.zeros_like(X)
    last_atom_mask[last_atom_index] = np.where(
        np.pad(is_append_iter, [[1, 0]], mode='constant', constant_values=1) == 1,
        np.ones_like(last_atom_index),
        np.ones_like(last_atom_index) * 2
    )
    return action_0, X, NX, A, NA, actions, last_atom_mask

In [22]:
def get_d(A, X):
    def _to_sparse(_A, _X):
        return sparse.coo_matrix(
            (np.ones([_A.shape[0] * 2], dtype=np.int32),
             (np.concatenate([_A[:, 0], _A[:, 1]], axis=0),
              np.concatenate([_A[:, 1], _A[:, 0]], axis=0))),
            shape=[_X.shape[0]] * 2)

    A_sparse = _to_sparse(A, X)
    d2 = A_sparse * A_sparse
    d3 = d2 * A_sparse

    D_2 = np.stack(d2.nonzero(), axis=1)
    D_2 = D_2[D_2[:, 0] < D_2[:, 1], :]

    D_3 = np.stack(d3.nonzero(), axis=1)
    D_3 = D_3[D_3[:, 0] < D_3[:, 1], :]

    D_3_sparse = _to_sparse(D_3, X)
    D_3_sparse = D_3_sparse - D_3_sparse.multiply(A_sparse)
    D_3 = np.stack(D_3_sparse.nonzero(), axis=1)
    D_3 = D_3[D_3[:, 0] < D_3[:, 1], :]
    return D_2, D_3


def merge_single_0(X_0, A_0, NX_0, NA_0):
    cumsum = np.cumsum(np.pad(NX_0, [[1, 0]], mode='constant')[:-1])
    A_0[:, :2] += np.stack([np.repeat(cumsum, NA_0)] * 2, axis=1)
    D_0_2, D_0_3 = get_d(A_0, X_0)

    A_split = []
    for i in range(get_mol_spec().num_bond_types):
        A_split.append(A_0[A_0[:, 2] == i, :2])
    A_split.extend([D_0_2, D_0_3])
    A_0 = A_split
    NX_rep_0 = np.repeat(np.arange(NX_0.shape[0]), NX_0)
    return X_0, A_0, NX_0, NX_rep_0


def merge_single(X, A, NX, NA, mol_ids, rep_ids, iw_ids, action_0, actions,
                 last_append_mask, log_p):
    X, A, NX, NX_rep = merge_single_0(X, A, NX, NA)
    cumsum = np.cumsum(np.pad(NX, [[1, 0]], mode='constant')[:-1])
    actions[:, -2] += cumsum * (actions[:, 0] == 0)
    actions[:, -1] += cumsum * (actions[:, 0] == 1)
    mol_ids_rep = np.repeat(mol_ids, NX)
    rep_ids_rep = np.repeat(rep_ids, NX)
    return (X, A, mol_ids_rep, rep_ids_rep, iw_ids, last_append_mask, NX, NX_rep,
            action_0, actions, log_p)

## Pair assembly

`ligands_*_corrected` and `proteins_*_corrected` are aligned line for line. Which file a row
sits in is a preprocessing artifact; the fold is defined by `<fold>_<tr|te>_unique_proteins.txt`,
so rows are pooled from both directories and routed by list membership.

In [23]:
def read_spaced(path, limit=None):
    out = []
    with open(path) as f:
        for line in f:
            s = ''.join(line.split())
            if s:
                out.append(s)
            if limit is not None and len(out) >= limit:
                break
    return out


def read_fasta(path):
    """[(name, sequence)] with the name taken up to the first whitespace, as prose does."""
    records, name, buf = [], None, []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                if name is not None:
                    records.append((name, "".join(buf)))
                name, buf = line[1:].split()[0], []
            elif line:
                buf.append(line)
    if name is not None:
        records.append((name, "".join(buf)))
    return records

In [24]:
def verify_fasta_order(split, unique):
    """The bepler FASTA must match the protein list record for record, in order.

    This is the invariant the whole conditioning path rests on: embeddings are keyed
    p1..pN by FASTA position, pairs are matched to rows by protein sequence in the list.
    A silent disagreement mis-conditions every ligand while every shape check passes.
    """
    d = split_dir(split)
    tag = "tr" if split == "train" else "te"
    fa = os.path.join(d, f"{FOLD}_{tag}_unique_proteins_bepler.fa")
    if not os.path.isfile(fa):
        raise FileNotFoundError(
            f"{fa} not found. It is the FASTA the embeddings were generated from, and the\n"
            f"  only way to confirm that HDF5 key p<i> really is row <i> of "
            f"{os.path.basename(d)}.\n"
            f"  Set VERIFY_FASTA_ORDER = False to skip this check (not recommended).")

    recs = read_fasta(fa)
    if len(recs) != len(unique):
        raise ValueError(
            f"{os.path.basename(fa)} has {len(recs)} records but the protein list has "
            f"{len(unique)}. They must correspond one-to-one, in order.")

    bad_name = [i for i, (n, _) in enumerate(recs) if n != f"p{i + 1}"]
    if bad_name:
        i = bad_name[0]
        raise ValueError(
            f"{os.path.basename(fa)}: record {i} is named {recs[i][0]!r}, expected "
            f"'p{i + 1}'. The HDF5 keys are matched by this name, so the order would be wrong.")

    mism = [i for i, ((_, s), u) in enumerate(zip(recs, unique))
            if "".join(s.split()).upper() != u.upper()]
    if mism:
        i = mism[0]
        raise ValueError(
            f"{os.path.basename(fa)} and the protein list disagree at {len(mism)} "
            f"position(s); first is record {i} (p{i + 1}):\n"
            f"  fasta: {recs[i][1][:60]}...\n"
            f"  list : {unique[i][:60]}...\n"
            f"  Every ligand would be conditioned on the wrong protein. Regenerate the "
            f"FASTA and the embeddings from the same protein list.")

    print(f"[{split}] order verified: {len(recs)} proteins, FASTA == protein list == p1..p{len(recs)}")

In [25]:
def read_pairs(split, limit=None):
    """Ligand/protein rows to assemble one split from. -> (ligands, proteins)

    The pair files do not correspond to the fold's protein split: for d1, 74 of the 104 test
    proteins have all their rows in the TRAIN pair file and only 30 in the test one. Which
    file a row sits in is a preprocessing artifact; the fold is defined by the unique protein
    lists, and assemble_pairs filters by those. So with POOL_PAIR_FILES the rows from every
    split directory are pooled first and routed by list membership afterwards. This cannot
    leak: the two lists are checked to share no protein.
    """
    sources = ("train", "test") if POOL_PAIR_FILES else (split,)
    seen, ligands, proteins = set(), [], []
    for src_split in sources:
        lig_path, prot_path = find_fold_files(src_split)[:2]
        lig = read_spaced(lig_path)
        prot = read_spaced(prot_path)
        if len(lig) != len(prot):
            raise ValueError(f"{src_split}: {len(lig)} ligand lines vs {len(prot)} protein "
                             f"lines - the pair files are not aligned.")
        n_new = 0
        for s, p in zip(lig, prot):
            key = (s, p.upper())
            if key in seen:
                continue
            seen.add(key)
            ligands.append(s)
            proteins.append(p)
            n_new += 1
        print(f"[{split}] + {n_new} new of {len(lig)} rows from "
              f"{os.path.basename(prot_path)}")
    if limit and len(ligands) > limit:
        # Sample across the pool rather than taking a prefix. The pooled rows are ordered
        # train-file-then-test-file, so a prefix can miss one split's proteins entirely.
        n_pooled = len(ligands)
        idx = sorted(random.Random(SPLIT_SEED).sample(range(n_pooled), limit))
        ligands = [ligands[i] for i in idx]
        proteins = [proteins[i] for i in idx]
        print(f"[{split}] LIMIT_PAIRS: sampled {limit} of {n_pooled} pooled rows "
              f"(SMOKE TEST - not the full dataset)")
    return ligands, proteins

In [26]:
def assemble_pairs(split, limit=None):
    """-> (records, C) where records = [(smiles, protein_row, ames), ...]"""
    lig_path, prot_path, uniq_path, emb_path = find_fold_files(split)
    print(f"[{split}] ligands   {os.path.basename(lig_path)}")
    print(f"[{split}] proteins  {os.path.basename(prot_path)}")
    print(f"[{split}] unique    {os.path.basename(uniq_path)}")
    print(f"[{split}] embedding {os.path.basename(emb_path)}")

    ligands, proteins = read_pairs(split, limit)

    unique = read_spaced(uniq_path)
    if not unique:
        raise ValueError(f"{uniq_path}: no protein sequences read.")
    # Upper-cased on both sides, as verify_fasta_order already compares.
    seq_to_row = {s.upper(): i for i, s in enumerate(unique)}
    if len(seq_to_row) != len(unique):
        dup = len(unique) - len(seq_to_row)
        print(f"[WARN] {os.path.basename(uniq_path)}: {dup} duplicate sequence(s); pairs "
              f"for a duplicate resolve to its LAST row.")

    # The embedding matrix is indexed positionally (row i == HDF5 key p{i+1} == FASTA
    # record i), while pairs are mapped by sequence content through the protein list.
    # Those two orderings are only equal if the FASTA and the list agree record for
    # record. If they ever drift, every ligand is conditioned on the wrong protein and
    # nothing downstream complains, so check it rather than assume it.
    if VERIFY_FASTA_ORDER:
        verify_fasta_order(split, unique)

    global N_C
    C = load_embeddings(emb_path, len(unique))
    N_C = int(C.shape[1])          # only differs from 6165 if ALLOW_N_C_OVERRIDE
    print(f"[{split}] {len(ligands)} pairs | {len(unique)} unique proteins | "
          f"embeddings {C.shape} (one {C.shape[1]}-d vector per protein)")

    # one parse per pair: validity, size cap, atom-type vocabulary
    rows, n_bad, n_big, n_oov, n_unmapped, n_long = [], 0, 0, 0, 0, 0
    for smi, seq in zip(ligands, proteins):
        row = seq_to_row.get(seq.upper())
        if row is None:
            n_unmapped += 1
            continue
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            n_bad += 1
            continue
        if mol.GetNumAtoms() > MAX_ATOMS:
            n_big += 1
            continue
        # graph_to_rnn is [batch, K, MAX_ITER]; a molecule needs GetNumBonds() + 1 steps, so
        # anything longer cannot be written into it and would abort the collator mid-run.
        if mol.GetNumBonds() + 1 > MAX_ITER:
            n_long += 1
            continue
        try:
            for a in mol.GetAtoms():
                get_mol_spec().get_atom_type(a)
        except ValueError:
            n_oov += 1
            continue
        rows.append((smi, row))

    print(f"[{split}] kept {len(rows)} | dropped: unparseable {n_bad}, "
          f">{MAX_ATOMS} atoms {n_big}, >{MAX_ITER - 1} bonds {n_long}, "
          f"out-of-vocab atoms {n_oov}, unmatched protein {n_unmapped}")
    kept_rows = {r for _, r in rows}
    unused = sorted(set(range(len(unique))) - kept_rows)
    if unused and limit is None:
        raise RuntimeError(
            f"{len(unused)} of the {len(unique)} proteins in "
            f"{os.path.basename(uniq_path)} never appear in {os.path.basename(prot_path)} "
            f"(first rows: {unused[:8]}). The list and the pair file describe different "
            f"populations, so this split cannot be assembled. Run diagnose_unmatched.py.")
    elif unused:
        # Expected under LIMIT_PAIRS. Harmless: the embedding matrix is indexed by position
        # in the full protein list, so proteins without sampled rows are simply unused.
        print(f"[{split}] [SMOKE] {len(unused)} of {len(unique)} proteins have no sampled "
              f"pair (LIMIT_PAIRS is set), so this split is a subset of the real one")

    if n_unmapped:
        frac = n_unmapped / max(1, len(ligands))
        print(f"[INFO] {n_unmapped} of {len(ligands)} pairs ({100 * frac:.1f}%) belong to "
              f"proteins outside this fold's split and were filtered out; all "
              f"{len(unique)} listed proteins are present.")
        if frac > UNMATCHED_TOLERANCE:
            raise RuntimeError(
                f"Dropped fraction {100 * frac:.1f}% exceeds UNMATCHED_TOLERANCE "
                f"({100 * UNMATCHED_TOLERANCE:.1f}%).")
    if not rows:
        raise RuntimeError(f"{split}: no usable pairs.")

    labels = ames_labels([smi for smi, _ in rows])
    records = [(smi, row, lab) for (smi, row), lab in zip(rows, labels)]
    return records, C

In [27]:
def split_by_protein(records, n_proteins, held_fraction, seed):
    """Split records by protein row so the two parts share no protein.

    -> (held_records, kept_records, held_rows, kept_rows)
    """
    rng = np.random.default_rng(seed)
    order = rng.permutation(n_proteins)
    n_held = max(1, int(round(held_fraction * n_proteins)))
    held_rows = set(order[:n_held].tolist())

    held = [r for r in records if r[1] in held_rows]
    kept = [r for r in records if r[1] not in held_rows]
    return held, kept, sorted(held_rows), sorted(set(order[n_held:].tolist()))

## Sampler and collator

Similar-length molecules batched together, K traversals each, `c` gathered per surviving record.

In [28]:
class BalancedSampler(Sampler):
    """Groups similar-length molecules into a batch to cut padding in the GCN/RNN.

    Upstream behaviour, kept: the records beyond the last whole batch (len % batch_size) are
    never sampled. The guard below is new - with fewer records than batch_size the sampler
    yields no batches at all, which silently produces an empty loader and a NaN validation
    loss rather than an error.
    """
    def __init__(self, cost, batch_size, name="dataset"):
        if len(cost) < batch_size:
            raise ValueError(
                f"{name}: {len(cost)} records but batch size {batch_size}. This sampler would "
                f"yield no batches. Lower the batch size, or widen LIMIT_PAIRS / "
                f"VAL_PROTEIN_FRACTION.")
        index = np.argsort(cost).tolist()
        chunk_size = int(float(len(cost)) / batch_size)
        dropped = len(cost) - chunk_size * batch_size
        if dropped:
            print(f"[INFO] {name}: {dropped} record(s) beyond the last whole batch are not "
                  f"sampled (upstream behaviour).")
        self.index = []
        for i in range(batch_size):
            self.index.append(index[i * chunk_size:(i + 1) * chunk_size])

    def _g(self):
        for index_i in self.index:
            random.shuffle(index_i)
        for batch_index in zip(*self.index):
            yield list(batch_index)

    def __iter__(self):
        return self._g()

    def __len__(self):
        return len(self.index[0])

In [29]:
def process_single(record, k=K_PATHS, p=P_ALPHA):
    """Build the graph, sample K traversals, attach the ligand's Ames label."""
    smiles, tox_label = record[0], int(record[2])

    graph, atom_types, atom_ranks, bonds, bond_types = get_graph_from_smiles(smiles)
    X_0 = np.array(atom_types, dtype=np.int32)
    A_0 = np.concatenate([
        np.array(bonds, dtype=np.int32),
        np.array(bond_types, dtype=np.int32)[:, np.newaxis]
    ], axis=1) if len(bonds) > 0 else np.zeros((0, 3), dtype=np.int32)

    X, A, NX, NA = [], [], [], []
    mol_ids, rep_ids, iw_ids = [], [], []
    action_0, actions, last_append_mask = [], [], []
    log_p = []

    tox_class = np.full((k,), tox_label, dtype=np.int32)

    for i in range(k):
        step_ids_i, log_p_i = traverse_graph(graph, atom_ranks, p=p)
        X_i, A_i = single_reorder(X_0, A_0, step_ids_i)
        action_0_i, X_i, NX_i, A_i, NA_i, actions_i, last_atom_mask_i = single_expand(X_i, A_i)

        X.append(X_i)
        A.append(A_i)
        NX.append(NX_i)
        NA.append(NA_i)
        action_0.append(action_0_i)
        actions.append(actions_i)
        last_append_mask.append(last_atom_mask_i)
        mol_ids.append(np.zeros_like(NX_i, dtype=np.int32))
        rep_ids.append(np.ones_like(NX_i, dtype=np.int32) * i)
        iw_ids.append(np.ones_like(NX_i, dtype=np.int32) * i)
        log_p.append(log_p_i)

    return (np.concatenate(X, axis=0), np.concatenate(A, axis=0),
            np.concatenate(NX, axis=0), np.concatenate(NA, axis=0),
            np.concatenate(mol_ids, axis=0), np.concatenate(rep_ids, axis=0),
            np.concatenate(iw_ids, axis=0), np.concatenate(action_0, axis=0),
            np.concatenate(actions, axis=0), np.concatenate(last_append_mask, axis=0),
            np.array(log_p, dtype=np.float32), tox_class)

In [30]:
class CMolRNNCollate:
    """Conditional collator. `c` is gathered for surviving records only, keeping its rows
    aligned with mol_ids."""
    def __init__(self, c_matrix, k=K_PATHS, p=P_ALPHA):
        self.C = c_matrix
        self.k = k
        self.p = p

    def __call__(self, batch):
        valid_batch = []
        for record in batch:
            try:
                _ = get_graph_from_smiles(record[0])
                valid_batch.append(record)
            except ValueError:
                continue
        if not valid_batch:
            return None

        shapes = [[0], [0, 3], [0], [0], [0], [0], [0], [0], [0, 5], [0], [0], [0]]
        dtypes = [np.int32] * 10 + [np.float32, np.int32]

        _build = lambda: [np.zeros(shape=s, dtype=d) for s, d in zip(shapes, dtypes)]
        _append = lambda r0, r1: [np.concatenate([a, b], axis=0) for a, b in zip(r0, r1)]

        (X, A, NX, NA, mol_ids, rep_ids, iw_ids, action_0, actions,
         last_append_mask, log_p, tox_class) = _build()

        for n, record in enumerate(valid_batch):
            (X_i, A_i, NX_i, NA_i, mol_ids_i, rep_ids_i, iw_ids_i, action_0_i,
             actions_i, last_append_mask_i, log_p_i, tox_class_i) = process_single(
                record, self.k, self.p)

            if n != 0:
                mol_ids_i += mol_ids[-1] + 1
                iw_ids_i += iw_ids[-1] + 1

            (X, A, NX, NA, mol_ids, rep_ids, iw_ids, action_0, actions,
             last_append_mask, log_p, tox_class) = _append(
                [X, A, NX, NA, mol_ids, rep_ids, iw_ids, action_0, actions,
                 last_append_mask, log_p, tox_class],
                [X_i, A_i, NX_i, NA_i, mol_ids_i, rep_ids_i, iw_ids_i, action_0_i,
                 actions_i, last_append_mask_i, log_p_i, tox_class_i.reshape(-1)])

        (X, A, mol_ids_rep, rep_ids_rep, iw_ids, last_append_mask, NX, NX_rep,
         action_0, actions, log_p) = merge_single(
            X, A, NX, NA, mol_ids, rep_ids, iw_ids, action_0, actions, last_append_mask, log_p)

        result_out = [X, A, mol_ids_rep, rep_ids_rep, iw_ids, last_append_mask, NX, NX_rep,
                      action_0, actions, log_p, tox_class]

        mol_list = [Chem.MolFromSmiles(b[0]) for b in valid_batch]
        graph_to_rnn = np.zeros((len(valid_batch), self.k, get_mol_spec().max_iter),
                                dtype=np.int32)
        rnn_to_graph = []
        cum_sum = 0
        for i, mol_i in enumerate(mol_list):
            num_iter = mol_i.GetNumBonds() + 1
            for k_idx in range(self.k):
                graph_to_rnn[i, k_idx, :num_iter] = np.arange(num_iter) + cum_sum
                r0 = np.ones([num_iter], dtype=np.int32) * i
                r1 = np.ones_like(r0) * k_idx
                r2 = np.arange(num_iter)
                rnn_to_graph.append(np.stack([r0, r1, r2], axis=0))
                cum_sum += num_iter
        rnn_to_graph = np.concatenate(rnn_to_graph, axis=1)
        NX_cum = np.cumsum(result_out[6])

        c = self.C[[rec[1] for rec in valid_batch], :]
        return result_out + [graph_to_rnn, rnn_to_graph, NX_cum, c]

In [31]:
def batch_to_tensor(record, device=None):
    """Move one collated batch to the device. `ids` maps each atom row to its molecule, which
    is how `c` is broadcast over atoms."""
    device = CTX if device is None else device
    *base, graph_to_rnn, rnn_to_graph, NX_cum, c = record
    (X, A, mol_ids_rep, rep_ids_rep, iw_ids, last_append_mask,
     NX, NX_rep, action_0, actions, log_p, tox_class) = base

    n_nodes = int(X.shape[0])
    X = torch.as_tensor(X, dtype=torch.long, device=device)

    A_sparse = []
    for A_i in A:
        if A_i.shape[0] == 0:
            A_sparse.append(None)
        else:
            A_i = np.concatenate([A_i, A_i[:, [1, 0]]], axis=0)
            idx = torch.as_tensor(A_i[:, :2].T.astype(np.int64), device=device)
            val = torch.ones(A_i.shape[0], dtype=torch.float32, device=device)
            A_sparse.append(torch.sparse_coo_tensor(
                idx, val, (n_nodes, n_nodes), dtype=torch.float32, device=device).coalesce())

    batch_size = int(mol_ids_rep.max() + 1)
    iw_size = int(rep_ids_rep.max() + 1)

    ids = torch.as_tensor(mol_ids_rep, dtype=torch.long, device=device)
    iw_ids, last_append_mask, NX, NX_rep, action_0, actions = [
        torch.as_tensor(x, dtype=torch.long, device=device)
        for x in [iw_ids, last_append_mask, NX, NX_rep, action_0, actions]]
    log_p = torch.as_tensor(log_p, dtype=torch.float32, device=device)
    tox_class = torch.as_tensor(tox_class, dtype=torch.float32, device=device)
    graph_to_rnn = torch.as_tensor(graph_to_rnn, dtype=torch.long, device=device)
    rnn_to_graph = torch.as_tensor(rnn_to_graph, dtype=torch.long, device=device)
    NX_cum = torch.as_tensor(NX_cum, dtype=torch.long, device=device)
    c = torch.as_tensor(np.ascontiguousarray(c), dtype=torch.float32, device=device)

    return [X, A_sparse, iw_ids, last_append_mask, NX, NX_rep, action_0, actions, log_p,
            batch_size, iw_size, tox_class, graph_to_rnn, rnn_to_graph, NX_cum, c, ids]

## Build datasets

The whole held-out split is the test set, matching the paper's 104 novel proteins. Early stopping
needs a validation set the test set cannot supply without contaminating it, and the paper
specifies none, so it is carved out of the train split by protein.

In [32]:
records_all_train, C_train = assemble_pairs("train", LIMIT_PAIRS)
records_test, C_held = assemble_pairs("test", LIMIT_PAIRS)

# Validation comes out of the train split, by protein; the whole held-out split is the test set.
records_val, records_train, val_rows, train_rows = split_by_protein(
    records_all_train, C_train.shape[0], VAL_PROTEIN_FRACTION, SPLIT_SEED)
test_rows = sorted({r[1] for r in records_test})

assert C_train.shape[1] == C_held.shape[1], (
    f"train embeddings are {C_train.shape[1]}-d but held-out are {C_held.shape[1]}-d")

# Protein-level leakage. The fold is built so train and held-out proteins share <20% sequence
# similarity; identity is the case that would be fatal, so check it rather than assume it.
_train_seqs = set(read_spaced(find_fold_files("train")[2]))
_held_seqs = set(read_spaced(find_fold_files("test")[2]))
_shared = _train_seqs & _held_seqs
if _shared:
    raise RuntimeError(
        f"{len(_shared)} protein sequence(s) are identical between the train and held-out "
        f"splits of fold {FOLD}. The held-out proteins would not be novel.")
assert not (set(val_rows) & set(train_rows)), "train and val share a protein row"

_shared_lig = {r[0] for r in records_train} & {r[0] for r in records_test}
if _shared_lig:
    print(f"[INFO] {len(_shared_lig)} ligand SMILES occur in both train and test, paired with "
          f"different proteins. The fold controls protein novelty, not ligand novelty.")

print(f"\ntrain {len(records_train)} pairs / {len(train_rows)} proteins")
print(f"val   {len(records_val)} pairs / {len(val_rows)} proteins  (carved from train)")
print(f"test  {len(records_test)} pairs / {len(test_rows)} proteins  (all held-out proteins)")

[train] ligands   ligands_train_4_org_corrected
[train] proteins  proteins_train_4_org_corrected
[train] unique    d1_tr_unique_proteins.txt
[train] embedding d1_tr_unique_embeddings_bepler.h5
[train] + 183964 new of 227093 rows from proteins_train_4_org_corrected
[train] + 9203 new of 11049 rows from proteins_test_4_org_corrected
[train] order verified: 1042 proteins, FASTA == protein list == p1..p1042
[train] 193167 pairs | 1042 unique proteins | embeddings (1042, 6165) (one 6165-d vector per protein)
[train] kept 125588 | dropped: unparseable 0, >100 atoms 0, >119 bonds 0, out-of-vocab atoms 20, unmatched protein 67559
[INFO] 67559 of 193167 pairs (35.0%) belong to proteins outside this fold's split and were filtered out; all 1042 listed proteins are present.
[INFO] Ames cache: 115116 entries from ligand_ames_labels.csv (fingerprint matches)
[INFO] Ames: 28056 toxic / 97532 non-toxic (22.3% toxic)
[test] ligands   ligands_test_4_org_corrected
[test] proteins  proteins_test_4_org_cor

In [33]:
# The held-out test set must cover every protein of the split, or the evaluation is not the
# paper's (Tables 1-3 are over all 104 novel proteins of d1). assemble_pairs already refuses
# to build a split with unused proteins when limit is None; this states the result.
n_held_proteins = int(C_held.shape[0])
if SMOKE_TEST:
    print(f"[SMOKE] test bundle covers {len(test_rows)}/{n_held_proteins} held-out proteins. "
          f"This is NOT the complete held-out test set - rerun with LIMIT_PAIRS = None "
          f"before reporting anything.")
elif len(test_rows) != n_held_proteins:
    raise RuntimeError(
        f"test bundle covers {len(test_rows)} of {n_held_proteins} held-out proteins. A full "
        f"run must include every one of them; this would be an incomplete test set.")
else:
    print(f"[OK] test bundle covers all {n_held_proteins} held-out proteins of fold {FOLD}.")

with open(SPLIT_MANIFEST, 'w') as f:
    json.dump({'fold': FOLD, 'fold_suffix': FOLD_SUFFIX[FOLD], 'N_C': N_C,
               'split_seed': SPLIT_SEED, 'val_protein_fraction': VAL_PROTEIN_FRACTION,
               'val_source': 'train split, by protein',
               'limit_pairs': LIMIT_PAIRS, 'smoke_test': SMOKE_TEST,
               'complete_held_out_test_set': not SMOKE_TEST,
               'n_held_out_proteins_total': n_held_proteins,
               'train_protein_rows': train_rows, 'val_protein_rows': val_rows,
               'test_protein_rows': test_rows,
               'n_train_pairs': len(records_train), 'n_val_pairs': len(records_val),
               'n_test_pairs': len(records_test)}, f, indent=1)

np.savez_compressed(
    TEST_BUNDLE,
    smiles=np.array([r[0] for r in records_test], dtype=object),
    protein_row=np.array([r[1] for r in records_test], dtype=np.int32),
    ames=np.array([r[2] for r in records_test], dtype=np.int8),
    embeddings=C_held,
    test_protein_rows=np.array(test_rows, dtype=np.int32),
    # Tags so a consumer of this file can tell a smoke subset from the real test set.
    smoke_test=np.array(SMOKE_TEST),
    limit_pairs=np.array(-1 if LIMIT_PAIRS is None else LIMIT_PAIRS),
    n_held_out_proteins_total=np.array(n_held_proteins))
print(f"[OK] test bundle -> {TEST_BUNDLE}")

[OK] test bundle covers all 104 held-out proteins of fold d1.


[OK] test bundle -> /home/muhamed/repo/CTDDG/outputs/finetune/d1/test_set.npz


In [34]:
# PORT: upstream `Lambda.__init__` shuffled with seed 17.
random.Random(17).shuffle(records_train)

loader_train = DataLoader(
    records_train,
    batch_sampler=BalancedSampler([len(r[0]) for r in records_train], BATCH_SIZE, "train"),
    num_workers=NUM_WORKERS,
    collate_fn=CMolRNNCollate(C_train, K_PATHS, P_ALPHA),
    persistent_workers=(NUM_WORKERS > 0))

loader_val = DataLoader(
    records_val,
    batch_sampler=BalancedSampler([len(r[0]) for r in records_val], BATCH_SIZE_VAL, "val"),
    num_workers=NUM_WORKERS,
    collate_fn=CMolRNNCollate(C_train, K_PATHS, P_ALPHA),   # val rows index C_train
    persistent_workers=(NUM_WORKERS > 0))

batch_cpu = next(iter(loader_train))
_b = batch_to_tensor(batch_cpu)
print(f"\nbatch: N={_b[9]} K={_b[10]} | X {tuple(_b[0].shape)} | "
      f"A blocks {len(_b[1])} | c {tuple(_b[15].shape)}")
assert _b[15].shape == (_b[9], N_C)
assert int(_b[16].max()) == _b[9] - 1

[INFO] train: 3 record(s) beyond the last whole batch are not sampled (upstream behaviour).
[INFO] val: 1 record(s) beyond the last whole batch are not sampled (upstream behaviour).

batch: N=8 K=5 | X (26456,) | A blocks 6 | c (8, 6165)


## Graph math

In [35]:
def segment_sum(X, idx, num_seg):
    out = torch.zeros((num_seg,) + tuple(X.shape[1:]), dtype=X.dtype, device=X.device)
    return out.index_add(0, idx, X)


def _concat_neighbourhoods(X, A_list):
    parts = [X]
    for A in A_list:
        parts.append(torch.sparse.mm(A, X) if A is not None else torch.zeros_like(X))
    return torch.cat(parts, dim=1)

In [36]:
class EfficientGraphConvFn(torch.autograd.Function):
    """Recomputes the concatenated neighbourhood in backward instead of storing it.
    Every A is symmetric, so A^T == A and backward reuses A directly."""

    @staticmethod
    def forward(ctx, X, W, A_list):
        ctx.A_list = A_list
        ctx.save_for_backward(X, W)
        return _concat_neighbourhoods(X, A_list) @ W

    @staticmethod
    @once_differentiable
    def backward(ctx, grad_output):
        X, W = ctx.saved_tensors
        A_list = ctx.A_list
        X_out = _concat_neighbourhoods(X, A_list)
        grad_output = grad_output.contiguous()

        grad_W = X_out.t() @ grad_output
        grad_X_out = grad_output @ W.t()

        chunks = torch.split(grad_X_out, X.shape[1], dim=1)
        grad_X = chunks[0]
        for A, g in zip(A_list, chunks[1:]):
            if A is not None:
                grad_X = grad_X + torch.sparse.mm(A, g.contiguous())
        return grad_X, grad_W, None


def get_activation(name):
    return {'relu': torch.relu, 'tanh': torch.tanh}[name]

## Layers

In [37]:
class Linear_BN(nn.Sequential):
    def __init__(self, F_in, F_out):
        super(Linear_BN, self).__init__(
            nn.Linear(F_in, F_out, bias=False),
            nn.BatchNorm1d(F_out, eps=1e-5, momentum=0.1))


class GraphConv(nn.Module):
    def __init__(self, Fin, Fout, D):
        super(GraphConv, self).__init__()
        self.Fin, self.Fout, self.D = Fin, Fout, D
        self.W = nn.Parameter(torch.empty(self.Fin * (self.D + 1), self.Fout))

    def forward(self, X, A_list):
        return EfficientGraphConvFn.apply(X, self.W, A_list)

In [38]:
class Policy(nn.Module):
    def __init__(self, F_in, F_h, N_A, N_B, k=1):
        super(Policy, self).__init__()
        self.F_in, self.F_h, self.N_A, self.N_B, self.k = F_in, F_h, N_A, N_B, k
        self.linear_h = Linear_BN(F_in * 2, self.F_h * k)
        self.linear_h_t = Linear_BN(F_in, self.F_h * k)
        self.linear_x = nn.Linear(self.F_h, self.N_B + self.N_B * self.N_A)
        self.linear_x_t = nn.Linear(self.F_h, 1)
        self.linear_pi = nn.Linear(self.F_in, self.k) if self.k > 1 else None

    def forward(self, X, NX, NX_rep, X_end=None):
        if X_end is None:
            X_end = segment_sum(X, NX_rep, NX.shape[0]) / NX.unsqueeze(1).float()
        X = torch.cat([X, X_end[NX_rep, :]], dim=1)

        X_h = torch.relu(self.linear_h(X)).reshape(-1, self.F_h)
        X_h_end = torch.relu(self.linear_h_t(X_end)).reshape(-1, self.F_h)

        # clip before exp() to stay inside float32, as in the original
        logits_x = torch.clamp(
            self.linear_x(X_h).reshape(-1, self.k, self.N_B + self.N_B * self.N_A), -80.0, 80.0)
        logits_x_t = torch.clamp(
            self.linear_x_t(X_h_end).reshape(-1, self.k, 1), -80.0, 80.0)

        X_x = torch.exp(logits_x)
        X_x_end = torch.exp(logits_x_t)

        # one joint softmax over every action of every atom plus `end`, per molecule
        X_sum = segment_sum(X_x, NX_rep, NX.shape[0]).sum(-1, keepdim=True) + X_x_end
        X_softmax = X_x / (X_sum[NX_rep, :, :] + 1e-10)
        X_softmax_end = X_x_end / (X_sum + 1e-10)

        if self.k > 1:
            pi = torch.softmax(self.linear_pi(X_end), dim=1).unsqueeze(-1)
            X_softmax = (X_softmax * pi[NX_rep, :, :]).sum(dim=1)
            X_softmax_end = (X_softmax_end * pi).sum(dim=1)
        else:
            X_softmax = X_softmax.squeeze(1)
            X_softmax_end = X_softmax_end.squeeze(1)

        connect, append = X_softmax[:, :self.N_B], X_softmax[:, self.N_B:]
        append = append.reshape(-1, self.N_A, self.N_B)
        return append, connect, X_softmax_end.squeeze(-1)

## Generator

Parameter names outside `linear_c.*` and `dense_policy_0.*` match the pretraining model, so the
stage-1 checkpoint transfers by name.

In [39]:
class MoleculeGenerator(nn.Module):
    __metaclass__ = ABCMeta

    def __init__(self, N_A, N_B, D, F_e, F_skip, F_c, Fh_policy, activation, *args, **kwargs):
        super(MoleculeGenerator, self).__init__()
        self.N_A, self.N_B, self.D = N_A, N_B, D
        self.F_e, self.F_skip, self.Fh_policy = F_e, F_skip, Fh_policy
        self.F_c = list(F_c)
        self.activation = get_activation(activation)

        self.embedding_atom = nn.Embedding(self.N_A, self.F_e)
        self.embedding_mask = nn.Embedding(3, self.F_e)
        self._build_graph_conv(*args, **kwargs)

        self.dense = nn.Sequential()
        for i, (f_in, f_out) in enumerate(zip([self.F_skip] + self.F_c[:-1], self.F_c)):
            self.dense.add_module(str(i), Linear_BN(f_in, f_out))

        self.policy_0 = nn.Parameter(torch.zeros(self.N_A))
        self.policy_h = Policy(self.F_c[-1], self.Fh_policy, self.N_A, self.N_B)
        self.mode = 'loss'
        self.last_l = None          # per-branch logging; always the UNFLOORED bound
        self.last_tox = None
        self.last_floored = None

    @abstractmethod
    def _build_graph_conv(self, *args, **kwargs): pass

    def _likelihood(self, init, append, connect, end, action_0, actions, iw_ids,
                    log_p_sigma, batch_size, iw_size, tox_class_batch):
        action_type, node_type, edge_type, append_pos, connect_pos = (
            actions[:, 0], actions[:, 1], actions[:, 2], actions[:, 3], actions[:, 4])

        def _log_mask(_x, _mask):
            return _mask * torch.log(_x + 1e-10) + (1 - _mask) * torch.zeros_like(_x)

        init = init.reshape(batch_size * iw_size, self.N_A)
        rows = torch.arange(action_0.shape[0], device=action_0.device)
        loss_init = torch.log(init[rows, action_0] + 1e-10)

        loss_end = _log_mask(end, (action_type == 2).to(end.dtype))
        loss_append = _log_mask(append[append_pos, node_type, edge_type],
                                (action_type == 0).to(end.dtype))
        loss_connect = _log_mask(connect[connect_pos, edge_type],
                                 (action_type == 1).to(end.dtype))

        log_p_x = loss_end + loss_append + loss_connect
        log_p_x = segment_sum(log_p_x.unsqueeze(-1), iw_ids, batch_size * iw_size).squeeze(-1)
        log_p_x = log_p_x + loss_init

        log_p_x = log_p_x.reshape(batch_size, iw_size)
        log_p_sigma = log_p_sigma.reshape(batch_size, iw_size)

        l = log_p_x - log_p_sigma
        l = torch.logsumexp(l, dim=1) - math.log(float(iw_size))

        # Eq. 8: all K paths share the molecule's label, so column 0 is the label
        if tox_class_batch.numel() != batch_size * iw_size:
            raise RuntimeError(f"tox_class has {tox_class_batch.numel()} entries, expected "
                               f"batch_size*K = {batch_size * iw_size}; labels would be "
                               f"misaligned with molecules.")
        tox_labels = tox_class_batch.reshape(batch_size, iw_size)[:, 0]

        # l_raw is the unfloored importance-sampled bound. Everything reported as a metric
        # (val_nll, l_nontoxic, l_toxic) must come from l_raw: if the floor leaked into them
        # the numbers would be a function of TOX_FLOOR, not of the model, and the divergence
        # warning would be watching the very quantity the floor pins down.
        l_raw = l
        if TOX_FLOOR is not None:
            l = torch.where(tox_labels > 0, torch.clamp(l, min=TOX_FLOOR), l)
            self.last_floored = ((tox_labels > 0) & (l_raw < TOX_FLOOR)).detach()
        else:
            self.last_floored = torch.zeros_like(tox_labels, dtype=torch.bool)
        l_weighted = l * (1.0 - (1.0 + LAMBDA_TOX) * tox_labels)

        self.last_l = l_raw.detach()
        self.last_tox = tox_labels.detach()
        return l_weighted

In [40]:
class MoleculeGenerator_RNN(MoleculeGenerator):
    __metaclass__ = ABCMeta

    def __init__(self, N_A, N_B, D, F_e, F_skip, F_c, Fh_policy, activation, N_rnn,
                 *args, **kwargs):
        super(MoleculeGenerator_RNN, self).__init__(
            N_A, N_B, D, F_e, F_skip, F_c, Fh_policy, activation, *args, **kwargs)
        self.N_rnn = N_rnn
        self.rnn = nn.GRU(input_size=self.F_c[-1] * 2, hidden_size=self.F_c[-1],
                          num_layers=self.N_rnn, batch_first=True)

    def _rnn_train(self, X, NX, NX_rep, graph_to_rnn, rnn_to_graph, NX_cum):
        X_avg = segment_sum(X, NX_rep, NX.shape[0]) / NX.unsqueeze(1).float()
        X = torch.cat([X_avg, X[NX_cum - 1]], dim=1)
        X = X[graph_to_rnn]
        batch_size, iw_size, length, num_features = X.shape
        out, _ = self.rnn(X.reshape(batch_size * iw_size, length, num_features))
        out = out.reshape(batch_size, iw_size, length, -1)
        return out[rnn_to_graph[0], rnn_to_graph[1], rnn_to_graph[2]]

    def _rnn_test(self, X, NX, NX_rep, NX_cum, h):
        X_avg = segment_sum(X, NX_rep, NX.shape[0]) / NX.unsqueeze(1).float()
        X = torch.cat([X_avg, X[NX_cum - 1]], dim=1)
        X, h = self.rnn(X.unsqueeze(1), h)
        return X.squeeze(1), h

In [41]:
class _TwoLayerDense(nn.Module):
    """FNN_0: protein embedding -> first-atom distribution."""
    def __init__(self, input_size, hidden_size, output_size):
        super(_TwoLayerDense, self).__init__()
        self.input = nn.Linear(input_size, hidden_size, bias=False)
        self.bn_input = nn.BatchNorm1d(hidden_size, eps=1e-5, momentum=0.1)
        self.output = nn.Linear(hidden_size, output_size, bias=True)

    def forward(self, c):
        return torch.softmax(self.output(torch.relu(self.bn_input(self.input(c)))), dim=-1)


class CMoleculeGenerator_RNN(MoleculeGenerator_RNN):
    __metaclass__ = ABCMeta

    def __init__(self, N_A, N_B, N_C, D, F_e, F_skip, F_c, Fh_policy, activation, N_rnn,
                 *args, **kwargs):
        self.N_C = N_C
        super(CMoleculeGenerator_RNN, self).__init__(
            N_A, N_B, D, F_e, F_skip, F_c, Fh_policy, activation, N_rnn, *args, **kwargs)
        self.dense_policy_0 = _TwoLayerDense(self.N_C, self.N_A * 3, self.N_A)

    @abstractmethod
    def _graph_conv_forward(self, X, A, c, ids): pass

    def _policy_0(self, c):
        # the 0.0 * policy_0 term keeps the unused pretraining parameter in the graph
        return self.dense_policy_0(c) + 0.0 * self.policy_0

    def _policy(self, X, A, NX, NX_rep, last_append_mask,
                graph_to_rnn, rnn_to_graph, NX_cum, c, ids):
        X = self.embedding_atom(X) + self.embedding_mask(last_append_mask)
        X = self._graph_conv_forward(X, A, c, ids)
        X_dense = self.dense(X)           # PORT: computed once, not twice as in MXNet
        X_mol = self._rnn_train(X_dense, NX, NX_rep, graph_to_rnn, rnn_to_graph, NX_cum)
        return self.policy_h(X_dense, NX, NX_rep, X_mol)

    def _decode_step(self, X, A, NX, NX_rep, last_append_mask, NX_cum, h, c, ids):
        """One sampling step — unused in training, needed by the sampler."""
        X = self.embedding_atom(X) + self.embedding_mask(last_append_mask)
        X = self._graph_conv_forward(X, A, c, ids)
        X_dense = self.dense(X)
        X_mol, h = self._rnn_test(X_dense, NX, NX_rep, NX_cum, h)
        append, connect, end = self.policy_h(X_dense, NX, NX_rep, X_mol)
        return append, connect, end, h

    def forward(self, *inputs):
        if self.mode in ('loss', 'likelihood'):
            (X, A, iw_ids, last_append_mask, NX, NX_rep, action_0, actions, log_p,
             batch_size, iw_size, tox_class, graph_to_rnn, rnn_to_graph, NX_cum, c, ids) = inputs
            init = self._policy_0(c).unsqueeze(1).expand(-1, iw_size, -1)
            append, connect, end = self._policy(
                X, A, NX, NX_rep, last_append_mask, graph_to_rnn, rnn_to_graph, NX_cum, c, ids)
            l_weighted = self._likelihood(init, append, connect, end, action_0, actions,
                                          iw_ids, log_p, batch_size, iw_size, tox_class)
            return l_weighted if self.mode == 'likelihood' else -l_weighted.mean()
        elif self.mode == 'decode_0':
            return self._policy_0(*inputs)
        elif self.mode == 'decode_step':
            return self._decode_step(*inputs)
        raise ValueError(f"Unknown mode: {self.mode}")

In [42]:
class CVanillaMolGen_RNN(CMoleculeGenerator_RNN):
    def __init__(self, N_A, N_B, N_C, D, F_e, F_h, F_skip, F_c, Fh_policy, activation, N_rnn):
        super(CVanillaMolGen_RNN, self).__init__(
            N_A, N_B, N_C, D, F_e, F_skip, F_c, Fh_policy, activation, N_rnn, F_h)

    def _build_graph_conv(self, F_h):
        self.F_h = list(F_h)
        self.conv = nn.ModuleList()
        self.bn = nn.ModuleList()            # bn[i-1] belongs to conv[i]; conv[0] has none
        for i, (f_in, f_out) in enumerate(zip([self.F_e] + self.F_h[:-1], self.F_h)):
            self.conv.append(GraphConv(f_in, f_out, self.N_B + self.D))
            if i != 0:
                self.bn.append(nn.BatchNorm1d(f_in, eps=1e-5, momentum=0.1))

        self.bn_skip = nn.BatchNorm1d(sum(self.F_h), eps=1e-5, momentum=0.1)
        self.linear_skip = Linear_BN(sum(self.F_h), self.F_skip)
        # FNN_1..FNN_l: the per-layer protein projectors
        self.linear_c = nn.ModuleList(
            [nn.Linear(self.N_C, f_out, bias=False) for f_out in self.F_h])

    def _graph_conv_forward(self, X, A, c, ids):
        X_out = [X]
        for i, conv in enumerate(self.conv):
            X = X_out[-1]
            c_i = self.linear_c[i](c)[ids, :]
            if i == 0:
                X_out.append(conv(X, A) + c_i)
            else:
                X_out.append(conv(self.activation(self.bn[i - 1](X)), A) + c_i)
        X_out = torch.cat(X_out[1:], dim=1)
        return self.activation(self.linear_skip(self.activation(self.bn_skip(X_out))))

## Init and early stopping

In [43]:
def init_weights_xavier(module):
    """MXNet's default Xavier (uniform, avg, magnitude 3) == xavier_uniform_ with gain 1."""
    for name, p in module.named_parameters():
        if p.dim() < 2 or 'bn' in name.split('.')[-1]:
            continue
        nn.init.xavier_uniform_(p, gain=1.0)
    for name, p in module.named_parameters():
        if 'bias' in name.split('.')[-1]:
            nn.init.zeros_(p)
    for m in module.modules():
        if isinstance(m, nn.BatchNorm1d):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)
            m.reset_running_stats()
    if hasattr(module, 'policy_0'):
        nn.init.zeros_(module.policy_0)
    return module


class EarlyStopping(object):
    def __init__(self, patience=5, delta=1e-2):
        self.patience, self.delta = patience, delta
        self.best, self.counter = math.inf, 0
        self.improved = False

    def step(self, loss):
        if loss < self.best - self.delta:
            self.best, self.counter, self.improved = loss, 0, True
        else:
            self.counter += 1
            self.improved = False
        return self.counter >= self.patience

    def state_dict(self):
        return {'best': self.best, 'counter': self.counter}

    def load_state_dict(self, s):
        self.best = s.get('best', self.best)
        self.counter = s.get('counter', self.counter)

## Model, weight transfer, resume

`ckpt_last.pt` is one file holding model, optimiser, counters, RNG streams and elapsed clock, so
the parts can never disagree about which iteration they came from. Written atomically, with a
rotated `.prev` fallback.

In [44]:
configs = {'N_C': N_C, 'F_e': F_E, 'F_h': F_H, 'F_skip': F_SKIP, 'F_c': F_C,
           'Fh_policy': FH_POLICY, 'activation': ACTIVATION, 'N_rnn': N_RNN}

def atomic_save(obj, path, keep_prev=True):
    """Write, fsync, rotate the old file to .prev, then rename into place.

    A plain torch.save() leaves a truncated file if the job is killed mid-write.
    """
    tmp = f"{path}.tmp.{os.getpid()}"
    try:
        with open(tmp, "wb") as fh:
            torch.save(obj, fh)
            fh.flush()
            os.fsync(fh.fileno())
        if keep_prev and os.path.exists(path):
            try:
                os.replace(path, path + ".prev")
            except OSError:
                pass
        os.replace(tmp, path)
        dfd = os.open(os.path.dirname(path) or ".", os.O_RDONLY)
        try:
            os.fsync(dfd)
        finally:
            os.close(dfd)
    finally:
        if os.path.exists(tmp):
            os.remove(tmp)

In [45]:
def robust_load(path):
    """Load `path`, falling back to its rotated .prev copy. -> (state, which) or (None, None)."""
    for candidate, which in ((path, "current"), (path + ".prev", "previous")):
        if not os.path.isfile(candidate):
            continue
        try:
            state = torch.load(candidate, map_location="cpu", weights_only=False)
            if which == "previous":
                print(f"[WARN] {os.path.basename(path)} was unreadable; recovered the rotated "
                      f"backup instead. At most one checkpoint interval is lost.")
            return state, which
        except Exception as e:
            print(f"[WARN] {os.path.basename(candidate)} unreadable ({type(e).__name__}: {e}).")
    return None, None

In [46]:
def rng_state():
    s = {"python": random.getstate(),
         "numpy": np.random.get_state(),
         "torch": torch.get_rng_state()}
    if torch.cuda.is_available():
        s["cuda"] = torch.cuda.get_rng_state_all()
    return s


def set_rng_state(s):
    """Restore the main-process RNG streams.

    This does not make a resumed run bit-identical to an uninterrupted one: the checkpoint is
    taken mid-epoch, and resuming restarts that epoch from its first batch.
    """
    if not s:
        return
    try:
        random.setstate(s["python"])
        np.random.set_state(s["numpy"])
        torch.set_rng_state(torch.as_tensor(s["torch"], dtype=torch.uint8))
        if torch.cuda.is_available() and "cuda" in s:
            torch.cuda.set_rng_state_all([torch.as_tensor(x, dtype=torch.uint8)
                                          for x in s["cuda"]])
    except Exception as e:
        print(f"[WARN] Could not restore RNG state ({type(e).__name__}); sampling continues "
              f"from a fresh stream. Training is unaffected apart from reproducibility.")

In [47]:
# ------------------------------------------------------------------ resume state
is_continuous = False
global_counter = 0
epoch_no = 1
resume_state = None
elapsed_offset = 0.0
prev_val_nll = math.inf
best_val_nll = math.inf

if FORCE_RESTART:
    # The previous run's checkpoints and logs were archived in the restart cell, before any
    # data work, so there is nothing to load here.
    print("[INFO] FORCE_RESTART is set - ignoring any existing checkpoint.")
else:
    resume_state, _which = robust_load(CKPT_LAST)
    if resume_state is not None and "optimizer" not in resume_state:
        # Checkpoint written by the older two-file scheme: pair it with trainer.pt.
        tstate, _ = robust_load(CKPT_TRAINER)
        if tstate is not None:
            resume_state = {**resume_state, **tstate}
            print("[INFO] Recovered a legacy ckpt_last.pt + trainer.pt pair.")
        else:
            print("[WARN] ckpt_last.pt has model weights but no optimizer state and no "
                  "trainer.pt beside it - restarting from the pretrained checkpoint instead.")
            resume_state = None
    is_continuous = resume_state is not None

In [48]:
def objective_signature():
    """Settings whose change makes a resumed run incomparable to the checkpoint.

    early_stop_on is here because early_stopping.best is a value of that metric. The split
    settings are here because changing them rebuilds the validation set: the restored best
    would belong to a different set, and proteins the model already trained on could land
    in validation.
    """
    return {'lambda_tox': LAMBDA_TOX, 'tox_floor': TOX_FLOOR,
            'early_stop_on': EARLY_STOP_ON, 'fold': FOLD, 'n_c': N_C,
            'split_seed': SPLIT_SEED, 'val_protein_fraction': VAL_PROTEIN_FRACTION}


def data_signature():
    """Settings that change the training set without invalidating the comparison."""
    return {'pool_pair_files': POOL_PAIR_FILES, 'limit_pairs': LIMIT_PAIRS,
            'max_atoms': MAX_ATOMS, 'max_iter': MAX_ITER, 'batch_size': BATCH_SIZE}

In [49]:
if is_continuous:
    configs = resume_state.get("configs") or (
        json.load(open(CKPT_CONFIG)) if os.path.isfile(CKPT_CONFIG) else configs)

    old_sig, new_sig = resume_state.get('objective'), objective_signature()
    if old_sig is None:
        print("[WARN] Checkpoint predates objective tracking; cannot confirm it was trained "
              "under the current objective and split settings.")
    else:
        # Compare only keys the checkpoint actually carries, so a checkpoint written before
        # a key was added resumes with a note rather than a spurious failure.
        changed = {k: (old_sig[k], v) for k, v in new_sig.items()
                   if k in old_sig and old_sig[k] != v}
        absent = [k for k in new_sig if k not in old_sig]
        if changed:
            raise RuntimeError(
                "Settings changed since the checkpoint: "
                + ", ".join(f"{k} {a!r} -> {b!r}" for k, (a, b) in changed.items())
                + "\n  Resuming would carry an optimiser state and an early-stopping best "
                  "that belong to the previous settings. Revert the change, or set "
                  "FORCE_RESTART = True to start a clean run under the new ones.")
        if absent:
            print(f"[WARN] Checkpoint does not record {absent}; cannot verify they are "
                  f"unchanged.")

    old_data, new_data = resume_state.get('data'), data_signature()
    if old_data:
        d = {k: (old_data[k], v) for k, v in new_data.items()
             if k in old_data and old_data[k] != v}
        # A smoke checkpoint and a full run are different experiments: different training
        # set, different validation set, incomparable epoch counters and early-stopping
        # best. This one is an error, not a warning.
        if 'limit_pairs' in d:
            a, b = d['limit_pairs']
            raise RuntimeError(
                f"The checkpoint was trained with LIMIT_PAIRS = {a!r}, this run uses {b!r}. "
                f"Resuming a smoke run as a full one (or the reverse) carries over an "
                f"optimiser state and an early-stopping best from a different dataset. "
                f"Set FORCE_RESTART = True for a clean run.")
        if d:
            print("[WARN] Training-set settings changed: "
                  + ", ".join(f"{k} {a!r} -> {b!r}" for k, (a, b) in d.items())
                  + ". Epoch boundaries and the iteration count are no longer comparable "
                    "with the earlier part of this run.")
    print("[INFO] Resuming an existing fine-tuning run.")
else:
    print("[INFO] Fresh fine-tuning run.")
    # Fresh without FORCE_RESTART: no checkpoint was found, or the one present was unusable.
    # Archive whatever is still on disk - checkpoints AND logs together - so this run cannot
    # append to the previous run's logs or be resumed from its leftovers. (The FORCE_RESTART
    # case already did this before data assembly.)
    archive_stale(RESUMABLE_ARTIFACTS, why="fresh run")

[INFO] Resuming an existing fine-tuning run.


In [50]:
if configs.get("N_C") != N_C:
    raise ValueError(
        f"Checkpoint was trained at N_C = {configs.get('N_C')} but this fold's embeddings are "
        f"{N_C}-d. The conditioning layers would not match. Use the matching embeddings, or "
        f"set FORCE_RESTART = True to start over at {N_C}.")

with open(CKPT_CONFIG, 'w') as f:
    json.dump(configs, f, indent=1)

model = CVanillaMolGen_RNN(get_mol_spec().num_atom_types, get_mol_spec().num_bond_types,
                           D=D_RECEPTIVE, **configs)

In [51]:
def load_pretrained(model, path):
    """Transfer phase-1 weights. Only the conditioning path may be missing."""
    state = torch.load(path, map_location='cpu', weights_only=False)
    sd = state['model'] if isinstance(state, dict) and 'model' in state else state
    missing, unexpected = model.load_state_dict(sd, strict=False)

    new = ('linear_c.', 'dense_policy_0.')
    unexplained = [k for k in missing if not k.startswith(new)]
    if unexplained:
        raise RuntimeError(f"Pretrained checkpoint missing non-conditioning tensors: "
                           f"{unexplained[:10]} — the two phases have drifted apart.")
    print(f"[INFO] Transferred {len(sd) - len(unexpected)}/{len(sd)} pretrained tensors; "
          f"{len(missing)} conditioning tensors newly initialised.")
    print("[INFO] A clean transfer means the shapes match - it is not evidence that the "
          "checkpoint was trained. See the provenance report above.")
    if unexpected:
        print(f"[WARN] Unused tensors in checkpoint: {unexpected[:10]}")


if is_continuous:
    model.load_state_dict(resume_state['model'])
else:
    init_weights_xavier(model)
    if os.path.exists(PRETRAINED_CKPT):
        load_pretrained(model, PRETRAINED_CKPT)
    elif REQUIRE_PRETRAINED:
        # Already raised in the config cell; kept as a backstop in case the file was removed
        # between then and now, or the cell was re-run with the flag flipped.
        raise FileNotFoundError(
            f"No pretrained checkpoint at {PRETRAINED_CKPT}.\n"
            f"  CTDDG is a two-phase method (paper 2.4): stage 1 pretrains unconditionally on "
            f"ChEMBL, stage 2 fine-tunes that model. Starting from random weights is a "
            f"different experiment and its numbers are not comparable to the paper.\n"
            f"  Run stage-1 pretraining first, or set REQUIRE_PRETRAINED = False to train "
            f"from scratch deliberately.")
    else:
        print("[WARN] REQUIRE_PRETRAINED is off - training from random init, which is NOT "
              "the two-phase protocol.")
model.to(CTX)

CVanillaMolGen_RNN(
  (embedding_atom): Embedding(65, 16)
  (embedding_mask): Embedding(3, 16)
  (conv): ModuleList(
    (0-5): 6 x GraphConv()
  )
  (bn): ModuleList(
    (0): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2-3): 2 x BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (4): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (bn_skip): BatchNorm1d(864, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (linear_skip): Linear_BN(
    (0): Linear(in_features=864, out_features=256, bias=False)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (linear_c): ModuleList(
    (0): Linear(in_features=6165, out_features=32, bias=False)
    (1): Linear(in_features=6165, out_features=64, bias=False)
    (2-3): 2 x Linear(in_features

In [52]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.0)
early_stopping = EarlyStopping(PATIENCE, ES_DELTA)
if PATIENCE >= MAX_EPOCHS:
    print(f"[WARN] PATIENCE={PATIENCE} >= MAX_EPOCHS={MAX_EPOCHS}: epoch 1 always improves "
          f"(best starts at inf), so the counter can reach at most {MAX_EPOCHS - 1} and early "
          f"stopping can never fire. Training will always run the full {MAX_EPOCHS} epochs. "
          f"This pair is inherited from the CDGCN reference; for early stopping to do "
          f"anything, PATIENCE must be < MAX_EPOCHS (2-3 is usual on a 10-epoch budget). "
          f"Left unchanged - it is a hyperparameter choice.")

[WARN] PATIENCE=10 >= MAX_EPOCHS=10: epoch 1 always improves (best starts at inf), so the counter can reach at most 9 and early stopping can never fire. Training will always run the full 10 epochs. This pair is inherited from the CDGCN reference; for early stopping to do anything, PATIENCE must be < MAX_EPOCHS (2-3 is usual on a 10-epoch budget). Left unchanged - it is a hyperparameter choice.


In [53]:
if is_continuous:
    optimizer.load_state_dict(resume_state['optimizer'])
    for g in optimizer.param_groups:
        g['lr'] = resume_state.get('lr', LR)
    global_counter = int(resume_state.get('global_counter', 0))
    epoch_no = int(resume_state.get('epoch_no', 1))
    early_stopping.load_state_dict(resume_state.get('early_stopping', {}))
    elapsed_offset = float(resume_state.get('elapsed_hours', 0.0))
    prev_val_nll = float(resume_state.get('prev_val_nll', math.inf))
    best_val_nll = float(resume_state.get('best_val_nll', math.inf))
    set_rng_state(resume_state.get('rng'))

    # Optimizer state tensors are loaded onto CPU; move them to the training device.
    for st in optimizer.state.values():
        for k, v in st.items():
            if torch.is_tensor(v):
                st[k] = v.to(CTX)

    print(f"[INFO] iteration {global_counter}, epoch {epoch_no}, "
          f"lr {optimizer.param_groups[0]['lr']:.3e}, best {early_stopping.best:.4f}, "
          f"patience {early_stopping.counter}/{PATIENCE}, "
          f"{elapsed_offset:.2f} h already spent")
    if epoch_no > MAX_EPOCHS:
        print(f"[INFO] This run already reached MAX_EPOCHS = {MAX_EPOCHS}. Raise it, or set "
              f"FORCE_RESTART = True, if you want to train further.")
    del resume_state
    gc.collect()

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_cond = sum(p.numel() for n, p in model.named_parameters()
             if n.startswith(('linear_c.', 'dense_policy_0.')))
print(f"[INFO] {n_trainable:,} trainable parameters ({n_cond:,} in the conditioning path)")

[INFO] iteration 120800, epoch 9, lr 8.906e-06, best 108.6283, patience 7/10, 2.18 h already spent
[INFO] 13,523,379 trainable parameters (6,541,865 in the conditioning path)


## Sanity checks

Eq. 8 against its closed form, the condition actually moving the loss, and the scale of the
conditioning term against the graph signal it is added to.

In [54]:
def sanity_check(model, batch_cpu):
    prev = model.mode
    model.mode = 'likelihood'
    model.eval()
    try:
        with torch.no_grad():
            inputs = batch_to_tensor(batch_cpu)
            lw = model(*inputs)
            l, tox = model.last_l, model.last_tox
        # last_l is the raw bound; the objective uses the floored one when TOX_FLOOR is set.
        l_eff = l if TOX_FLOOR is None else torch.where(
            tox > 0, torch.clamp(l, min=TOX_FLOOR), l)
        err = (lw - l_eff * (1.0 - (1.0 + LAMBDA_TOX) * tox)).abs().max().item()
        n_tox = int(tox.sum())
        print(f"Eq. 8  | {tox.numel()} molecules, {n_tox} toxic | max error {err:.2e}")
        assert err < 1e-3, "Eq. 8 weighting does not match the closed form."
        if n_tox:
            flipped = int((torch.sign(lw[tox > 0]) == -torch.sign(l_eff[tox > 0])).sum())
            print(f"       | sign flipped on {flipped}/{n_tox} toxic molecules")
            if TOX_FLOOR is not None:
                print(f"       | {int(model.last_floored.sum())}/{n_tox} toxic molecules "
                      f"already at the floor")

        model.mode = 'loss'
        with torch.no_grad():
            a = model(*inputs).item()
            # Swap on a COPY: inputs[15] is the real ProSE embedding and the scale
            # diagnostic below reads it. Mutating it in place made that diagnostic
            # measure the random tensor instead.
            shuffled = list(inputs)
            shuffled[15] = torch.randn_like(inputs[15]) * 0.1
            b = model(*shuffled).item()
        print(f"cond   | real c {a:.4f} vs random c {b:.4f} | delta {abs(a - b):.4f}")
        assert abs(a - b) > 1e-6, "The condition has no effect — conditioning path is dead."

        # Scale of the additive condition against the graph signal it is added to. The
        # conditioning path is ~48% of the parameters and is freshly initialised, so if
        # linear_c(c) dominates conv(X, A) the pretrained graph weights are being swamped.
        with torch.no_grad():
            X, A, c, ids = inputs[0], inputs[1], inputs[15], inputs[16]
            h = model.embedding_atom(X) + model.embedding_mask(inputs[3])
            print(f"scale  | std(c) = {c.std().item():.3f}")
            worst = 0.0
            for i, conv in enumerate(model.conv):
                inp = h if i == 0 else model.activation(model.bn[i - 1](h))
                g = conv(inp, A)
                c_i = model.linear_c[i](c)[ids, :]
                r = c_i.std().item() / max(1e-9, g.std().item())
                worst = max(worst, r)
                print(f"       | layer {i}: std(conv) {g.std().item():7.3f}  "
                      f"std(c_i) {c_i.std().item():7.3f}  ratio {r:6.1f}x")
                h = g + c_i
            if worst > 10:
                print(f"       | [WARN] the condition is up to {worst:.0f}x the graph signal. "
                      f"This matches upstream CDGCN's initialisation, but at this ratio the "
                      f"pretrained graph weights contribute little early in fine-tuning.")
        print("[OK]")
    finally:
        model.mode = prev
        model.train()

In [55]:
sanity_check(model, batch_cpu)

Eq. 8  | 8 molecules, 0 toxic | max error 0.00e+00
cond   | real c 47.6993 vs random c 46.9627 | delta 0.7366
scale  | std(c) = 0.057
       | layer 0: std(conv)   2.276  std(c_i)   0.273  ratio    0.1x
       | layer 1: std(conv)   4.387  std(c_i)   0.387  ratio    0.1x
       | layer 2: std(conv)   5.364  std(c_i)   0.582  ratio    0.1x
       | layer 3: std(conv)  10.023  std(c_i)   0.699  ratio    0.1x
       | layer 4: std(conv)   1.745  std(c_i)   0.634  ratio    0.4x
       | layer 5: std(conv)   0.360  std(c_i)   0.674  ratio    1.9x
[OK]


## Training loop

In [56]:
TRAIN_HEADER = ('step\ttime(h)\tloss\tlr\tl_nontoxic\tl_toxic\tepoch\tfrac_toxic'
                '\tfrac_floored\n')
VAL_HEADER = ('step\ttime(h)\tepoch\tval_weighted\tval_nll\tval_l_nontoxic\tval_l_toxic'
              '\tes_counter\tbest\n')


def _is_data_row(line):
    line = line.strip()
    return bool(line) and not line.startswith(('#', 'step\t'))


def ensure_header(path, header):
    """Create the log with its header if absent or empty. Never truncates.

    A fresh run's previous logs were archived alongside its checkpoints, so there is nothing
    here to clear; truncating instead would destroy the log of the run in progress whenever
    this cell is re-executed on its own.
    """
    if not os.path.isfile(path) or os.path.getsize(path) == 0:
        with open(path, 'w') as f:
            f.write(header)


if not is_continuous:
    t0 = time.time()
    ensure_header(LOG_FILE, TRAIN_HEADER)
    ensure_header(VAL_LOG_FILE, VAL_HEADER)
else:
    for path, header in ((LOG_FILE, TRAIN_HEADER), (VAL_LOG_FILE, VAL_HEADER)):
        ensure_header(path, header)
    # Prefer the checkpoint's own elapsed time; the log is the fallback for legacy runs.
    last_hours = elapsed_offset
    if last_hours <= 0.0:
        try:
            with open(LOG_FILE) as f:
                rows = [r.strip() for r in f if _is_data_row(r)]
            if rows:
                last_hours = float(rows[-1].split('\t')[1])
        except (OSError, ValueError, IndexError):
            pass
    t0 = time.time() - last_hours * 3600.0
    print(f"[INFO] Clock resumed at {last_hours:.2f} h.")

[INFO] Clock resumed at 2.18 h.


In [57]:
def evaluate(model, loader):
    """Molecule-weighted means: (Eq. 8 loss, plain NLL, mean l non-toxic, mean l toxic)."""
    prev = model.mode
    model.mode = 'likelihood'
    model.eval()
    sum_w = sum_l = 0.0
    n_mol = 0
    sum_non, n_non, sum_tox, n_tox = 0.0, 0, 0.0, 0
    try:
        with torch.no_grad():
            for batch in loader:
                if batch is None:
                    continue
                lw = model(*batch_to_tensor(batch))
                l, tox = model.last_l, model.last_tox
                sum_w += float(lw.sum())
                sum_l += float(l.sum())
                n_mol += int(l.numel())
                if (tox == 0).any():
                    sum_non += float(l[tox == 0].sum()); n_non += int((tox == 0).sum())
                if (tox > 0).any():
                    sum_tox += float(l[tox > 0].sum()); n_tox += int((tox > 0).sum())
    finally:
        model.mode = prev
        model.train()
    if n_mol == 0:
        raise RuntimeError("Validation loader produced no molecules; early stopping would "
                           "silently see NaN. Check BATCH_SIZE_VAL against the val split size.")
    return (-sum_w / n_mol, -sum_l / n_mol,
            sum_non / n_non if n_non else float('nan'),
            sum_tox / n_tox if n_tox else float('nan'))

In [58]:
def die_nonfinite(what, value):
    """Stop on a numerical failure WITHOUT touching ckpt_last.pt.

    The previous version checkpointed here, which overwrote the last good state with the
    one that had just failed - exactly the state you do not want to keep.
    """
    path = os.path.join(OUT_DIR, f"nonfinite_{time.strftime('%Y%m%d-%H%M%S')}.pt")
    try:
        atomic_save({'model': model.state_dict(), 'global_counter': global_counter,
                     'epoch_no': epoch_no, 'what': what}, path, keep_prev=False)
    except Exception as e:
        print(f"[WARN] could not write {path}: {type(e).__name__}")
    raise RuntimeError(
        f"Non-finite {what} ({value}) at iteration {global_counter}. ckpt_last.pt is "
        f"untouched, so re-running resumes from the last good state; the failing weights "
        f"are in {os.path.basename(path)} for inspection. Note this is a real numerical "
        f"failure, unlike an 'nan' in the l_toxic column, which only means a logging "
        f"window held no toxic molecule.")

In [59]:
def save_state(best=False, reason=""):
    """One self-contained, atomically written checkpoint."""
    state = {
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'lr': optimizer.param_groups[0]['lr'],
        'global_counter': global_counter,
        'epoch_no': epoch_no,
        'early_stopping': early_stopping.state_dict(),
        'prev_val_nll': prev_val_nll,
        'best_val_nll': best_val_nll,
        'elapsed_hours': (time.time() - t0) / 3600.0,
        'rng': rng_state(),
        'configs': configs,
        'fold': FOLD,
        'N_C': N_C,
        'lambda_tox': LAMBDA_TOX,
        'objective': objective_signature(),
        'data': data_signature(),
        'smoke_test': SMOKE_TEST,
        'saved_at': time.strftime('%Y-%m-%d %H:%M:%S'),
        'reason': reason,
    }
    atomic_save(state, CKPT_LAST)

    # trainer.pt is kept only so older tooling that reads it still works.
    atomic_save({k: state[k] for k in ('optimizer', 'lr', 'global_counter', 'epoch_no',
                                       'early_stopping')}, CKPT_TRAINER, keep_prev=False)

    if best:
        atomic_save({'model': model.state_dict(), 'configs': configs, 'fold': FOLD,
                     'epoch': epoch_no - 1, 'global_counter': global_counter,
                     'val_loss': early_stopping.best,
                     'val_metric': EARLY_STOP_ON,     # which quantity val_loss is
                     'N_C': N_C, 'lambda_tox': LAMBDA_TOX,
                     'objective': objective_signature(),
                     'smoke_test': SMOKE_TEST,        # not a model to report numbers from
                     'saved_at': state['saved_at']}, CKPT_BEST)

In [60]:
# SIGTERM (SLURM time limit) / SIGINT (interrupt kernel) checkpoint at the next iteration
# boundary. A second signal falls through to the default handler.
_stop = {'requested': False, 'name': None}


def _request_stop(signum, frame):
    if _stop['requested']:
        signal.signal(signum, signal.SIG_DFL)
        os.kill(os.getpid(), signum)
        return
    _stop['requested'] = True
    _stop['name'] = signal.Signals(signum).name
    print(f"\n[SIGNAL] {_stop['name']} received - finishing this iteration, checkpointing, "
          f"then exiting. Send it again to abort immediately.", flush=True)


if GRACEFUL_SIGNALS:
    for _sig in (signal.SIGTERM, signal.SIGINT):
        try:
            signal.signal(_sig, _request_stop)
        except (ValueError, OSError):
            pass        # not on the main thread; fall back to default handling


def done():
    """Training is over when the epoch budget is spent or early stopping has triggered."""
    if epoch_no > MAX_EPOCHS:
        return f"MAX_EPOCHS = {MAX_EPOCHS} reached"
    if early_stopping.counter >= PATIENCE:
        return f"early stopping already triggered ({early_stopping.counter}/{PATIENCE})"
    return None

In [61]:
model.mode = 'loss'
model.train()
it_train = iter(loader_train)
interrupted = False
win = {'loss': 0.0, 'n_batch': 0, 'sum_non': 0.0, 'n_non': 0, 'sum_tox': 0.0, 'n_tox': 0,
       'n_floored': 0}

if done():
    print(f"Nothing to do: {done()}. Raise MAX_EPOCHS/PATIENCE or set FORCE_RESTART = True.")
else:
    print(f"Training from iteration {global_counter}, epoch {epoch_no}."
          + ("  [SMOKE TEST - results are not reportable]" if SMOKE_TEST else ""))

Training from iteration 120800, epoch 9.


In [62]:
while not done():
    try:
        inputs_cpu = next(it_train)
    except StopIteration:
        # Epoch boundary. Validate here, before any step of the next epoch, so the reported
        # epoch is exactly the model that finished it.
        finished = epoch_no
        epoch_no += 1
        it_train = iter(loader_train)

        val_w, val_nll, val_non, val_tox = evaluate(model, loader_val)
        stop = early_stopping.step(val_w if EARLY_STOP_ON == 'weighted' else val_nll)
        hours = (time.time() - t0) / 3600.0
        with open(VAL_LOG_FILE, 'a') as f:
            f.write(f"{global_counter}\t{hours:.4f}\t{finished}\t{val_w:.4f}"
                    f"\t{val_nll:.4f}\t{val_non:.4f}\t{val_tox:.4f}"
                    f"\t{early_stopping.counter}\t{early_stopping.best:.4f}\n")
        print(f"epoch {finished} | val weighted {val_w:.4f} | nll {val_nll:.4f} | "
              f"l_nontox {val_non:.3f} | l_tox {val_tox:.3f} | best {early_stopping.best:.4f} | "
              f"patience {early_stopping.counter}/{PATIENCE}"
              f"{'  <- new best' if early_stopping.improved else ''}")
        # Watch plain NLL against its own best, independently of the weighted objective.
        # The previous check only looked when the weighted loss improved, so it stayed silent
        # through a run where BOTH branches degraded.
        if val_nll < best_val_nll:
            best_val_nll = val_nll
        elif val_nll > best_val_nll + NLL_DIVERGENCE_MARGIN:
            print(f"  [WARN] val NLL {val_nll:.2f} is {val_nll - best_val_nll:.2f} above its "
                  f"best ({best_val_nll:.2f}). Eq. 8's toxic branch is unbounded below, so the "
                  f"model can keep lowering toxic likelihood at the cost of everything else. "
                  f"If l_tox keeps falling while l_nontox also worsens, stop and set "
                  f"TOX_FLOOR (see the config cell).")
        prev_val_nll = val_nll

        save_state(best=early_stopping.improved, reason=f"epoch{finished}")
        if stop:
            print(f"Early stopping after epoch {finished}.")
        continue

    if inputs_cpu is None:          # whole batch dropped by the OOV shield
        continue
    global_counter += 1

    optimizer.zero_grad(set_to_none=True)        # PORT: Gluon overwrites grads, torch accumulates
    loss = model(*batch_to_tensor(inputs_cpu))
    if not torch.isfinite(loss):
        die_nonfinite("loss", loss.item())
    loss.backward()

    # clip_grad_value_ clamps, and clamping NaN returns NaN - a non-finite gradient would
    # pass straight through into optimizer.step() and poison every parameter, after which
    # the loss check above only fires once the damage is done. One reduction, one sync.
    _grads = [p.grad for p in model.parameters() if p.grad is not None]
    if _grads and not torch.isfinite(torch.stack([g.sum() for g in _grads]).sum()):
        die_nonfinite("gradient", float('nan'))

    torch.nn.utils.clip_grad_value_(model.parameters(), CLIP_GRAD)   # MXNet clip_gradient
    optimizer.step()

    if global_counter % DECAY_STEP == 0:
        for g in optimizer.param_groups:
            g['lr'] *= (1.0 - DECAY)

    if global_counter % CHECKPOINT_EVERY == 0:
        save_state(reason=f"periodic@{global_counter}")

    # Accumulate over the logging window. Reporting a single batch made l_tox read 'nan'
    # whenever that batch happened to hold no toxic molecule - at batch 8 and ~22% toxic that
    # is 0.78^8 = 13% of batches, which is noise, not divergence.
    _l, _tox = model.last_l, model.last_tox          # l is the raw, unfloored bound
    win['loss'] += loss.item(); win['n_batch'] += 1
    win['sum_non'] += float(_l[_tox == 0].sum()); win['n_non'] += int((_tox == 0).sum())
    win['sum_tox'] += float(_l[_tox > 0].sum()); win['n_tox'] += int((_tox > 0).sum())
    win['n_floored'] += int(model.last_floored.sum())

    if global_counter % SUMMARY_STEP == 0:
        lr_now = optimizer.param_groups[0]['lr']
        hours = (time.time() - t0) / 3600.0
        mean_loss = win['loss'] / max(1, win['n_batch'])
        l_non = win['sum_non'] / win['n_non'] if win['n_non'] else float('nan')
        l_tox = win['sum_tox'] / win['n_tox'] if win['n_tox'] else float('nan')
        frac_tox = win['n_tox'] / max(1, win['n_tox'] + win['n_non'])
        frac_floor = win['n_floored'] / max(1, win['n_tox'])

        with open(LOG_FILE, 'a') as f:
            f.write(f"{global_counter}\t{hours:.4f}\t{mean_loss:.4f}\t{lr_now}"
                    f"\t{l_non:.4f}\t{l_tox:.4f}\t{epoch_no}\t{frac_tox:.4f}"
                    f"\t{frac_floor:.4f}\n")
        print(f"iter {global_counter:06d} | ep {epoch_no} | loss {mean_loss:8.3f} | "
              f"l_nontox {l_non:8.2f} | l_tox {l_tox:9.2f} | tox {100 * frac_tox:4.1f}%"
              + (f" | floored {100 * frac_floor:5.1f}%" if TOX_FLOOR is not None else "")
              + f" | lr {lr_now:.2e} | {hours:.2f}h")
        if TOX_FLOOR is not None and win['n_tox'] and frac_floor > 0.98:
            print("  [WARN] essentially every toxic molecule is at the floor: the toxic "
                  "branch contributes no gradient, which is equivalent to LAMBDA_TOX = 0. "
                  "Lower the floor or accept that toxic ligands are simply being ignored.")
        for k in win:
            win[k] = 0.0
        del loss, inputs_cpu
        gc.collect()

    # Honour a pending SIGTERM/SIGINT on an iteration boundary, with the weights consistent.
    if _stop['requested']:
        save_state(reason=f"signal:{_stop['name']}@{global_counter}")
        with open(LOG_FILE, 'a') as f:
            f.write(f"# interrupted by {_stop['name']} at iteration {global_counter}, "
                    f"epoch {epoch_no}\n")
        print(f"[SIGNAL] Checkpointed at iteration {global_counter} (epoch {epoch_no}). "
              f"Re-run this notebook to continue from here.")
        interrupted = True
        break

iter 121000 | ep 9 | loss   -9.858 | l_nontox   -78.53 | l_tox   -346.17 | tox 20.8% | lr 8.87e-06 | 2.19h
iter 121200 | ep 9 | loss  -20.715 | l_nontox   -76.56 | l_tox   -355.76 | tox 22.5% | lr 8.84e-06 | 2.19h
iter 121400 | ep 9 | loss  -13.167 | l_nontox   -76.51 | l_tox   -313.39 | tox 23.0% | lr 8.80e-06 | 2.20h
iter 121600 | ep 9 | loss  -32.148 | l_nontox   -83.87 | l_tox   -382.53 | tox 24.9% | lr 8.76e-06 | 2.20h
iter 121800 | ep 9 | loss  -23.977 | l_nontox   -79.55 | l_tox   -369.35 | tox 23.1% | lr 8.73e-06 | 2.20h
iter 122000 | ep 9 | loss  -11.390 | l_nontox   -74.67 | l_tox   -338.82 | tox 20.8% | lr 8.69e-06 | 2.21h
iter 122200 | ep 9 | loss  -24.768 | l_nontox   -75.03 | l_tox   -397.37 | tox 21.1% | lr 8.66e-06 | 2.21h
iter 122400 | ep 9 | loss  -23.249 | l_nontox   -73.29 | l_tox   -351.06 | tox 22.8% | lr 8.63e-06 | 2.21h
iter 122600 | ep 9 | loss  -16.184 | l_nontox   -79.05 | l_tox   -340.70 | tox 22.7% | lr 8.59e-06 | 2.22h
iter 122800 | ep 9 | loss  -15.498 | 

In [63]:
if not interrupted:
    save_state(reason="final")
    if not os.path.exists(CKPT_BEST):
        save_state(best=True)   # no epoch improved; keep the final weights as "best"
    with open(LOG_FILE, 'a') as f:
        f.write('# finished\n')
    print(f"\nDone. Best validation {EARLY_STOP_ON} loss: {early_stopping.best:.4f}")
else:
    print(f"\nStopped early by {_stop['name']} at iteration {global_counter}. "
          f"Best validation {EARLY_STOP_ON} loss so far: {early_stopping.best:.4f}")

if SMOKE_TEST:
    print("[SMOKE TEST] This run used LIMIT_PAIRS; its checkpoints, test bundle and metrics "
          "are pipeline artifacts, not results.")
for name, path in (("best weights", CKPT_BEST), ("last weights", CKPT_LAST),
                   ("configs", CKPT_CONFIG), ("test bundle", TEST_BUNDLE),
                   ("split manifest", SPLIT_MANIFEST), ("train log", LOG_FILE),
                   ("val log", VAL_LOG_FILE)):
    print(f"  {name:15s} {path}")


Done. Best validation nll loss: 108.6283
  best weights    /home/muhamed/repo/CTDDG/outputs/finetune/d1/ckpt_best.pt
  last weights    /home/muhamed/repo/CTDDG/outputs/finetune/d1/ckpt_last.pt
  configs         /home/muhamed/repo/CTDDG/outputs/finetune/d1/configs.json
  test bundle     /home/muhamed/repo/CTDDG/outputs/finetune/d1/test_set.npz
  split manifest  /home/muhamed/repo/CTDDG/outputs/finetune/d1/split_manifest.json
  train log       /home/muhamed/repo/CTDDG/outputs/finetune/d1/log.out
  val log         /home/muhamed/repo/CTDDG/outputs/finetune/d1/val.out
